# MGD ML Model Preprocessing: All

In [1]:
import pandas as pd
from pathlib import Path
from Bio import SeqIO
from Bio.SeqUtils import GC
from sklearn.preprocessing import OneHotEncoder
import csv

In [2]:
# Data directory: download raw input data from Zenodo (see inputData/README.md)
# and extract into this location before running this notebook.
DATA_DIR = Path("../inputData/maize_inputData")

# Load data
maize_df = pd.read_csv(DATA_DIR / "4578_duplicated_pairs_v4.csv")
display(maize_df)

,Maize1,Location1,Maize2,Location2,Group
0,Zm00001d034914,arms,Zm00001d012815,arms,I
1,Zm00001d034896,arms,Zm00001d012817,arms,I
2,Zm00001d034890,arms,Zm00001d012820,arms,I
3,Zm00001d034886,arms,Zm00001d012823,arms,I
4,Zm00001d034885,arms,Zm00001d012827,arms,I
...,...,...,...,...,...
4573,Zm00001d046979,peri,Zm00001d036635,arms,IV
4574,Zm00001d046981,peri,Zm00001d036638,arms,IV
4575,Zm00001d046986,peri,Zm00001d036623,arms,IV
4576,Zm00001d046996,peri,Zm00001d036626,arms,IV


## Histone Markers

In [3]:
histone_file_paths = [
    ('Zm-B73_v4_genes_2kbdown_40wins_H2AZ_log_ave', 'H2AZ_down'),
    ('Zm-B73_v4_genes_2kbdown_40wins_H3K4me1_log_ave', 'H3K4me1_down'),
    ('Zm-B73_v4_genes_2kbdown_40wins_H3K4me3_log_ave', 'H3K4me3_down'),
    ('Zm-B73_v4_genes_2kbdown_40wins_H3K9ac_log_ave', 'H3K9ac_down'),
    ('Zm-B73_v4_genes_2kbdown_40wins_H3K27ac_log_ave', 'H3K27ac_down'),
    ('Zm-B73_v4_genes_2kbdown_40wins_H3K27me3_log_ave', 'H3K27me3_down'),
    ('Zm-B73_v4_genes_2kbdown_40wins_H3K36me3_log_ave', 'H3K36me3_down'),
    ('Zm-B73_v4_genes_2kbdown_40wins_H3K56ac_log_ave', 'H3K56ac_down'),

    ('Zm-B73_v4_genes_genebody_40wins_H2AZ_log_ave', 'H2AZ_genebody'),
    ('Zm-B73_v4_genes_genebody_40wins_H3K4me1_log_ave', 'H3K4me1_genebody'),
    ('Zm-B73_v4_genes_genebody_40wins_H3K4me3_log_ave', 'H3K4me3_genebody'),
    ('Zm-B73_v4_genes_genebody_40wins_H3K9ac_log_ave', 'H3K9ac_genebody'),
    ('Zm-B73_v4_genes_genebody_40wins_H3K27ac_log_ave', 'H3K27ac_genebody'),
    ('Zm-B73_v4_genes_genebody_40wins_H3K27me3_log_ave', 'H3K27me3_genebody'),
    ('Zm-B73_v4_genes_genebody_40wins_H3K36me3_log_ave', 'H3K36me3_genebody'),
    ('Zm-B73_v4_genes_genebody_40wins_H3K56ac_log_ave', 'H3K56ac_genebody'),

    ('Zm-B73_v4_genes_2kbup_40wins_H2AZ_log_ave', 'H2AZ_up'),
    ('Zm-B73_v4_genes_2kbup_40wins_H3K4me1_log_ave', 'H3K4me1_up'),
    ('Zm-B73_v4_genes_2kbup_40wins_H3K4me3_log_ave', 'H3K4me3_up'),
    ('Zm-B73_v4_genes_2kbup_40wins_H3K9ac_log_ave', 'H3K9ac_up'),
    ('Zm-B73_v4_genes_2kbup_40wins_H3K27ac_log_ave', 'H3K27ac_up'),
    ('Zm-B73_v4_genes_2kbup_40wins_H3K27me3_log_ave', 'H3K27me3_up'),
    ('Zm-B73_v4_genes_2kbup_40wins_H3K36me3_log_ave', 'H3K36me3_up'),
    ('Zm-B73_v4_genes_2kbup_40wins_H3K56ac_log_ave', 'H3K56ac_up'),
]

### Histone Data Pre-check

In [4]:
import pandas as pd
import os

# All unique gene IDs from master list
master_genes = set(maize_df['Maize1']).union(set(maize_df['Maize2']))
print("Total duplicated genes count: ", len(list(master_genes)))

print("=== Histone Data Pre-check ===\n")
total_warnings = 0

for file_path, col_name in histone_file_paths:
    df_tmp = pd.read_csv(DATA_DIR / file_path, delimiter='\t', header=None, names=['raw_id', col_name])
    df_tmp['gene_id'] = df_tmp['raw_id'].str.split('_').str[0]
    
    # Check for duplicate gene IDs after parsing
    duplicates = df_tmp['gene_id'].duplicated()
    if duplicates.any():
        dup_ids = df_tmp.loc[duplicates, 'gene_id'].unique().tolist()
        print(f"DUPLICATE gene IDs detected in {col_name}: {len(dup_ids)} duplicates")
        print(f"   Affected IDs: {dup_ids[:10]}{'...' if len(dup_ids) > 10 else ''}")
        total_warnings += 1

    # Check for master list genes absent from this file
    file_genes = set(df_tmp['gene_id'])
    absent_genes = master_genes - file_genes
    
    if absent_genes:
        print(f"{col_name}: {len(absent_genes)} master list genes absent from file")
        print(f"   Absent gene IDs: {sorted(list(absent_genes))[:10]}{'...' if len(absent_genes) > 10 else ''}")
        total_warnings += 1
    else:
        print(f"  ✓  {col_name}: all master list genes present")

print(f"\n=== Pre-check complete. Total warnings: {total_warnings} ===")
print("NaNs will be introduced for absent genes during merge.")

Total duplicated genes count:  9156
=== Histone Data Pre-check ===

  ✓  H2AZ_down: all master list genes present
  ✓  H3K4me1_down: all master list genes present
  ✓  H3K4me3_down: all master list genes present
  ✓  H3K9ac_down: all master list genes present
  ✓  H3K27ac_down: all master list genes present
  ✓  H3K27me3_down: all master list genes present
  ✓  H3K36me3_down: all master list genes present
  ✓  H3K56ac_down: all master list genes present
  ✓  H2AZ_genebody: all master list genes present
  ✓  H3K4me1_genebody: all master list genes present
  ✓  H3K4me3_genebody: all master list genes present
  ✓  H3K9ac_genebody: all master list genes present
  ✓  H3K27ac_genebody: all master list genes present
  ✓  H3K27me3_genebody: all master list genes present
  ✓  H3K36me3_genebody: all master list genes present
  ✓  H3K56ac_genebody: all master list genes present
  ✓  H2AZ_up: all master list genes present
  ✓  H3K4me1_up: all master list genes present
  ✓  H3K4me3_up: all master l

### Histone Processing and Merge

In [5]:
print("=== Histone Data Processing and Merge ===\n")

histone_frames = []

for file_path, col_name in histone_file_paths:
    df_tmp = pd.read_csv(DATA_DIR / file_path, delimiter='\t', header=None, names=['raw_id', col_name])
    df_tmp['gene_id'] = df_tmp['raw_id'].str.split('_').str[0]
    
    # Deduplicate: keep first occurrence, consistent with genome-wide files
    df_tmp = df_tmp.drop_duplicates(subset='gene_id', keep='first')
    df_tmp = df_tmp[['gene_id', col_name]].set_index('gene_id')
    
    histone_frames.append(df_tmp)

# Build single wide histone dataframe
histone_wide = pd.concat(histone_frames, axis=1).reset_index()
histone_wide = histone_wide.rename(columns={'index': 'gene_id'})

print(f"Histone wide dataframe shape: {histone_wide.shape}")
print(f"Expected columns: {1 + len(histone_file_paths)} (gene_id + 24 histone features)")

# Merge M1
histone_M1 = histone_wide.copy()
histone_M1 = histone_M1.rename(columns={col: f'{col}_M1' for _, col in histone_file_paths})
histone_M1 = histone_M1.rename(columns={'gene_id': 'Maize1'})
maize_df = pd.merge(maize_df, histone_M1, on='Maize1', how='left')

# Merge M2
histone_M2 = histone_wide.copy()
histone_M2 = histone_M2.rename(columns={col: f'{col}_M2' for _, col in histone_file_paths})
histone_M2 = histone_M2.rename(columns={'gene_id': 'Maize2'})
maize_df = pd.merge(maize_df, histone_M2, on='Maize2', how='left')

# Hard checkpoint
assert len(maize_df) == 4578, f"CRITICAL: Row count is {len(maize_df)}, expected 4578. Silent duplication occurred."
print(f"\n✓ Row count verified: {len(maize_df)} rows")
print(f"✓ Column count: {len(maize_df.columns)}")

# NaN report for histone columns
histone_cols = [f'{col}_{suffix}' for _, col in histone_file_paths for suffix in ['M1', 'M2']]
nan_report = maize_df[histone_cols].isna().sum()
nan_report = nan_report[nan_report > 0]
if len(nan_report) > 0:
    print(f"\n⚠️  NaNs detected in histone columns (from absent genes flagged in pre-check):")
    print(nan_report)
else:
    print("\n✓ No NaNs in histone columns")

=== Histone Data Processing and Merge ===

Histone wide dataframe shape: (39492, 25)
Expected columns: 25 (gene_id + 24 histone features)

✓ Row count verified: 4578 rows
✓ Column count: 53

✓ No NaNs in histone columns


In [6]:
display(maize_df)

,Maize1,Location1,Maize2,Location2,Group,H2AZ_down_M1,H3K4me1_down_M1,H3K4me3_down_M1,H3K9ac_down_M1,H3K27ac_down_M1,...,H3K36me3_genebody_M2,H3K56ac_genebody_M2,H2AZ_up_M2,H3K4me1_up_M2,H3K4me3_up_M2,H3K9ac_up_M2,H3K27ac_up_M2,H3K27me3_up_M2,H3K36me3_up_M2,H3K56ac_up_M2
0,Zm00001d034914,arms,Zm00001d012815,arms,I,-0.026707,-0.060370,-0.080652,-0.034977,0.055285,...,1.898475,0.041880,-0.045810,-0.059693,-0.048348,-0.011720,0.006547,-0.008638,-0.071359,-0.027517
1,Zm00001d034896,arms,Zm00001d012817,arms,I,0.847482,-0.053267,-0.103033,-0.069541,0.054290,...,1.415251,0.181027,-0.034041,-0.063115,-0.053382,-0.008865,-0.025941,0.003964,-0.095701,-0.038679
2,Zm00001d034890,arms,Zm00001d012820,arms,I,-0.019175,-0.088925,-0.078468,-0.023988,-0.088444,...,1.037050,0.161094,0.149880,-0.075306,-0.022485,-0.078616,-0.024235,-0.049216,-0.131599,-0.045503
3,Zm00001d034886,arms,Zm00001d012823,arms,I,0.060965,0.541438,0.078854,0.019759,0.124945,...,1.006114,0.161962,0.032570,0.080465,0.388075,0.074593,0.248631,-0.033331,0.209014,0.263999
4,Zm00001d034885,arms,Zm00001d012827,arms,I,0.145319,0.029047,0.501670,0.089114,0.375345,...,1.153594,0.460089,-0.048856,-0.108711,-0.068237,-0.091855,-0.020913,-0.068742,-0.119929,-0.064639
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4573,Zm00001d046979,peri,Zm00001d036635,arms,IV,-0.057408,-0.057085,-0.057615,0.035318,-0.057781,...,1.833603,0.474707,0.011068,-0.005385,-0.019787,0.046278,0.009533,0.015416,-0.014929,0.003539
4574,Zm00001d046981,peri,Zm00001d036638,arms,IV,0.000384,-0.023365,0.008214,0.027345,-0.015147,...,-0.097733,0.029449,0.666150,-0.045887,0.008584,0.086127,0.189805,1.090263,-0.107230,0.087639
4575,Zm00001d046986,peri,Zm00001d036623,arms,IV,0.539172,0.068531,0.431359,0.160578,0.490966,...,-0.010617,0.112401,-0.021885,-0.105206,-0.012539,-0.052353,0.013274,-0.054500,-0.109377,-0.069578
4576,Zm00001d046996,peri,Zm00001d036626,arms,IV,-0.106654,0.559587,0.530210,0.109033,0.229105,...,1.137396,1.000991,-0.022015,-0.044374,0.012846,0.047267,0.099781,-0.019617,-0.023711,-0.014908


## GC Content: Gene and Promoter

In [7]:
# gc_gene_promoter_output.csv is generated by
# maize_scripts/feature_preprocessing/gc_calc_prom_gene_processor.py
# (see that script for its required raw inputs)
gc_content = pd.read_csv(DATA_DIR / "gc_gene_promoter_output.csv")
print(gc_content.columns)
display(gc_content)

Index(['Gene_ID', 'Transcript_ID', 'Promoter_GC', 'Gene_CDS_GC',
       'Promoter_Source'],
      dtype='object')


,Gene_ID,Transcript_ID,Promoter_GC,Gene_CDS_GC,Promoter_Source
0,Zm00001d022017,Zm00001d022017_T001,0.66,0.687805,Jores
1,Zm00001d021788,Zm00001d021788_T001,0.65,0.598214,Jores
2,Zm00001d021419,Zm00001d021419_T001,0.34,0.421405,Jores
3,Zm00001d021685,Zm00001d021685_T022,0.37,0.433096,Jores
4,Zm00001d020731,Zm00001d020731_T001,0.47,0.687654,Jores
...,...,...,...,...,...
9151,Zm00001d013554,Zm00001d013554_T001,0.61,0.693252,Jores
9152,Zm00001d017876,Zm00001d017876_T018,0.58,0.439051,Jores
9153,Zm00001d013038,Zm00001d013038_T002,0.49,0.668038,Jores
9154,Zm00001d017833,Zm00001d017833_T001,0.45,0.547035,Jores


### GC Content Pre-check

In [8]:
import pandas as pd

print("=== GC Content Pre-check ===\n")
print(f"Input dataframe shape: {gc_content.shape}")
print(f"Columns: {gc_content.columns.tolist()}\n")

# Hard duplicate check — this should never happen with primary transcript data
duplicates = gc_content['Gene_ID'].duplicated()
if duplicates.any():
    dup_ids = gc_content.loc[duplicates, 'Gene_ID'].unique().tolist()
    raise ValueError(
        f"CRITICAL: {len(dup_ids)} duplicate Gene_IDs detected in GC input file. "
        f"This indicates an upstream processing error that must be corrected before proceeding.\n"
        f"Affected IDs: {dup_ids}"
    )
else:
    print("✓ No duplicate Gene_IDs detected\n")

# Cross-reference against master list
master_genes = set(maize_df['Maize1']).union(set(maize_df['Maize2']))
file_genes = set(gc_content['Gene_ID'])
absent_genes = master_genes - file_genes

if absent_genes:
    print(f"WARNING: {len(absent_genes)} master list genes absent from GC input file.")
    print(f"   These will produce NaN values in the merge and require pair-wise dropout.")
    print(f"   Absent gene IDs: {sorted(list(absent_genes))}")
else:
    print("✓ All master list genes present in GC input file\n")

print("=== Pre-check complete. Review warnings before proceeding. ===")

=== GC Content Pre-check ===

Input dataframe shape: (9156, 5)
Columns: ['Gene_ID', 'Transcript_ID', 'Promoter_GC', 'Gene_CDS_GC', 'Promoter_Source']

✓ No duplicate Gene_IDs detected

✓ All master list genes present in GC input file

=== Pre-check complete. Review warnings before proceeding. ===


### GC Content Processing and Merge

In [9]:
print("=== GC Content Processing and Merge ===\n")

# Select and rename relevant columns
gc_clean = gc_content[['Gene_ID', 'Gene_CDS_GC', 'Promoter_GC']].rename(columns={
    'Gene_CDS_GC': 'GC_genic',
    'Promoter_GC': 'GC_prom'
})

# Merge M1
gc_M1 = gc_clean.rename(columns={
    'Gene_ID': 'Maize1',
    'GC_genic': 'GC_genic_M1',
    'GC_prom': 'GC_prom_M1'
})
maize_df = pd.merge(maize_df, gc_M1, on='Maize1', how='left')

# Merge M2
gc_M2 = gc_clean.rename(columns={
    'Gene_ID': 'Maize2',
    'GC_genic': 'GC_genic_M2',
    'GC_prom': 'GC_prom_M2'
})
maize_df = pd.merge(maize_df, gc_M2, on='Maize2', how='left')

# Hard checkpoint
assert len(maize_df) == 4578, f"CRITICAL: Row count is {len(maize_df)}, expected 4578. Silent duplication occurred."
print(f"✓ Row count verified: {len(maize_df)} rows")
print(f"✓ Column count: {len(maize_df.columns)}")

# NaN report
gc_cols = ['GC_genic_M1', 'GC_prom_M1', 'GC_genic_M2', 'GC_prom_M2']
nan_report = maize_df[gc_cols].isna().sum()
nan_report = nan_report[nan_report > 0]
if len(nan_report) > 0:
    print(f"\n NaNs detected in GC columns:")
    print(nan_report)
else:
    print("\n✓ No NaNs in GC columns")

=== GC Content Processing and Merge ===

✓ Row count verified: 4578 rows
✓ Column count: 57

✓ No NaNs in GC columns


In [10]:
display(maize_df)

,Maize1,Location1,Maize2,Location2,Group,H2AZ_down_M1,H3K4me1_down_M1,H3K4me3_down_M1,H3K9ac_down_M1,H3K27ac_down_M1,...,H3K4me3_up_M2,H3K9ac_up_M2,H3K27ac_up_M2,H3K27me3_up_M2,H3K36me3_up_M2,H3K56ac_up_M2,GC_genic_M1,GC_prom_M1,GC_genic_M2,GC_prom_M2
0,Zm00001d034914,arms,Zm00001d012815,arms,I,-0.026707,-0.060370,-0.080652,-0.034977,0.055285,...,-0.048348,-0.011720,0.006547,-0.008638,-0.071359,-0.027517,0.426833,0.550000,0.416449,0.39
1,Zm00001d034896,arms,Zm00001d012817,arms,I,0.847482,-0.053267,-0.103033,-0.069541,0.054290,...,-0.053382,-0.008865,-0.025941,0.003964,-0.095701,-0.038679,0.444239,0.610000,0.471496,0.60
2,Zm00001d034890,arms,Zm00001d012820,arms,I,-0.019175,-0.088925,-0.078468,-0.023988,-0.088444,...,-0.022485,-0.078616,-0.024235,-0.049216,-0.131599,-0.045503,0.651584,0.647059,0.656250,0.62
3,Zm00001d034886,arms,Zm00001d012823,arms,I,0.060965,0.541438,0.078854,0.019759,0.124945,...,0.388075,0.074593,0.248631,-0.033331,0.209014,0.263999,0.554924,0.500000,0.523040,0.58
4,Zm00001d034885,arms,Zm00001d012827,arms,I,0.145319,0.029047,0.501670,0.089114,0.375345,...,-0.068237,-0.091855,-0.020913,-0.068742,-0.119929,-0.064639,0.520384,0.590000,0.509479,0.62
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4573,Zm00001d046979,peri,Zm00001d036635,arms,IV,-0.057408,-0.057085,-0.057615,0.035318,-0.057781,...,-0.019787,0.046278,0.009533,0.015416,-0.014929,0.003539,0.420114,0.620000,0.428288,0.55
4574,Zm00001d046981,peri,Zm00001d036638,arms,IV,0.000384,-0.023365,0.008214,0.027345,-0.015147,...,0.008584,0.086127,0.189805,1.090263,-0.107230,0.087639,0.608491,0.510000,0.579276,0.44
4575,Zm00001d046986,peri,Zm00001d036623,arms,IV,0.539172,0.068531,0.431359,0.160578,0.490966,...,-0.012539,-0.052353,0.013274,-0.054500,-0.109377,-0.069578,0.804762,0.470000,0.804938,0.55
4576,Zm00001d046996,peri,Zm00001d036626,arms,IV,-0.106654,0.559587,0.530210,0.109033,0.229105,...,0.012846,0.047267,0.099781,-0.019617,-0.023711,-0.014908,0.609195,0.560000,0.599415,0.54


## Expression Data 

In [11]:
# Expression_Features_LogTau.csv is generated by
# maize_scripts/feature_preprocessing/expression_log2_tau_processing.py
# from raw Expression_correct.csv
exp_df = pd.read_csv(DATA_DIR / "Expression_Features_LogTau.csv")
print(exp_df.columns)
display(exp_df)

Index(['gene_id', 'Log2_Average', 'Tau_Index'], dtype='object')


,gene_id,Log2_Average,Tau_Index
0,Zm00001d027230,1.336389,0.471082
1,Zm00001d027231,4.217901,0.348919
2,Zm00001d027232,0.034814,0.950473
3,Zm00001d027233,0.024654,0.708662
4,Zm00001d027234,0.004819,0.979057
...,...,...,...
39000,Zm00001d026706,4.405282,0.234547
39001,Zm00001d026707,0.019758,1.000000
39002,Zm00001d026709,0.000000,NaN
39003,Zm00001d026711,0.030612,0.948881


### Expression Data Pre-check

In [12]:
import pandas as pd

print("=== Expression Data Pre-check ===\n")
print(f"Input dataframe shape: {exp_df.shape}")
print(f"Columns: {exp_df.columns.tolist()}\n")

# Hard duplicate check
duplicates = exp_df['gene_id'].duplicated()
if duplicates.any():
    dup_ids = exp_df.loc[duplicates, 'gene_id'].unique().tolist()
    raise ValueError(
        f"CRITICAL: {len(dup_ids)} duplicate gene_ids detected in expression input file. "
        f"This indicates an upstream processing error that must be corrected before proceeding.\n"
        f"Affected IDs: {dup_ids}"
    )
else:
    print("✓ No duplicate gene_ids detected\n")

# Cross-reference against master list
master_genes = set(maize_df['Maize1']).union(set(maize_df['Maize2']))
file_genes = set(exp_df['gene_id'])
absent_genes = master_genes - file_genes

if absent_genes:
    print(f"WARNING: {len(absent_genes)} master list genes absent from expression file.")
    print(f"   These will produce NaN in both Log2_Average and Tau_Index and require pair-wise dropout.")
    print(f"   Absent gene IDs: {sorted(list(absent_genes))}\n")
else:
    print("✓ All master list genes present in expression file\n")

# Check for NaN in Log2_Average — this is real missing data requiring pair-wise dropout
nan_log2_mask = exp_df['Log2_Average'].isna()
if nan_log2_mask.any():
    nan_log2_genes = exp_df.loc[nan_log2_mask, 'gene_id'].tolist()
    print(f"WARNING: {len(nan_log2_genes)} genes have NaN in Log2_Average.")
    print(f"   These genes and their homoeologs will be dropped from the final dataset.")
    print(f"   Affected gene IDs: {nan_log2_genes}\n")
else:
    print("✓ No NaN values detected in Log2_Average\n")

# Report Tau NaNs separately — these are biologically meaningful and will be retained
nan_tau_mask = exp_df['Tau_Index'].isna()
if nan_tau_mask.any():
    nan_tau_genes = exp_df.loc[nan_tau_mask, 'gene_id'].tolist()
    print(f"INFO: {len(nan_tau_genes)} genes have NaN in Tau_Index (FPKM = 0 across all tissues).")
    print(f"   These NaN values are biologically meaningful and will be RETAINED in the final dataset.")
    print()
else:
    print("✓ No NaN values in Tau_Index\n")

print("=== Pre-check complete. Review warnings before proceeding. ===")

=== Expression Data Pre-check ===

Input dataframe shape: (39005, 3)
Columns: ['gene_id', 'Log2_Average', 'Tau_Index']

✓ No duplicate gene_ids detected

✓ All master list genes present in expression file

✓ No NaN values detected in Log2_Average

INFO: 2811 genes have NaN in Tau_Index (FPKM = 0 across all tissues).
   These NaN values are biologically meaningful and will be RETAINED in the final dataset.

=== Pre-check complete. Review warnings before proceeding. ===


### Expression Data Processing and Merge

In [13]:
print("=== Expression Data Processing and Merge ===\n")

# Select and rename relevant columns
exp_clean = exp_df[['gene_id', 'Log2_Average', 'Tau_Index']]

# Merge M1
exp_M1 = exp_clean.rename(columns={
    'gene_id': 'Maize1',
    'Log2_Average': 'avg_expression_M1',
    'Tau_Index': 'tau_M1'
})
maize_df = pd.merge(maize_df, exp_M1, on='Maize1', how='left')

# Merge M2
exp_M2 = exp_clean.rename(columns={
    'gene_id': 'Maize2',
    'Log2_Average': 'avg_expression_M2',
    'Tau_Index': 'tau_M2'
})
maize_df = pd.merge(maize_df, exp_M2, on='Maize2', how='left')

# Hard checkpoint
assert len(maize_df) == 4578, f"CRITICAL: Row count is {len(maize_df)}, expected 4578. Silent duplication occurred."
print(f"✓ Row count verified: {len(maize_df)} rows")
print(f"✓ Column count: {len(maize_df.columns)}")

# NaN report — distinguish between acceptable and problematic NaNs
print("\n--- NaN Report ---")
for col in ['avg_expression_M1', 'avg_expression_M2']:
    n = maize_df[col].isna().sum()
    if n > 0:
        print(f"{col}: {n} NaNs — real missing data, pairs will be dropped at final merge step")
    else:
        print(f"✓  {col}: no NaNs")

for col in ['tau_M1', 'tau_M2']:
    n = maize_df[col].isna().sum()
    if n > 0:
        print(f"{col}: {n} NaNs — biologically meaningful (FPKM = 0), will be RETAINED")
    else:
        print(f"✓  {col}: no NaNs")

=== Expression Data Processing and Merge ===

✓ Row count verified: 4578 rows
✓ Column count: 61

--- NaN Report ---
✓  avg_expression_M1: no NaNs
✓  avg_expression_M2: no NaNs
tau_M1: 66 NaNs — biologically meaningful (FPKM = 0), will be RETAINED
tau_M2: 82 NaNs — biologically meaningful (FPKM = 0), will be RETAINED


#### Note: NaNs Retained
NaNs from tau_M1 and tau_M2 are retained: where all FPKM values across tissues are zero, the Tau Index is mathematically undefined. This is a real biological state.

## Ka, Ks, and Omega for Genes

In [14]:
KaKsO = pd.read_csv(DATA_DIR / "KaKsO-Analysis-recombination-20200724.csv", index_col=False)
print(KaKsO.columns)
display(KaKsO)

Index(['Maize1', 'Chr._M1', 'Start_M1', 'End_M1', 'Ka_M1', 'Ks_M1', 'O_M1',
       'Loc_M1', 'Maize2', 'Chr._M2', 'Start_M2', 'End_M2', 'Ka_M2', 'Ks_M2',
       'O_M2', 'Sorghum syntelog', 'Unnamed: 16'],
      dtype='object')


,Maize1,Chr._M1,Start_M1,End_M1,Ka_M1,Ks_M1,O_M1,Loc_M1,Maize2,Chr._M2,Start_M2,End_M2,Ka_M2,Ks_M2,O_M2,Sorghum syntelog,Unnamed: 16
0,Zm00001d034914,1,"305,341,236","305,352,529",0.0095,0.1597,0.0595,arms,Zm00001d012815,5,"776,419","787,466",0.0211,0.1668,0.1264,Sobic.001G004400,arms
1,Zm00001d034896,1,"305,008,203","305,016,567",0.1012,0.0651,1.5547,arms,Zm00001d012817,5,"834,692","841,621",0.0163,0.1557,0.1044,Sobic.001G005600,arms
2,Zm00001d034890,1,"304,868,663","304,870,630",0.0397,0.7272,0.0546,arms,Zm00001d012820,5,"848,901","851,867",0.0300,0.5350,0.0561,Sobic.001G006400,arms
3,Zm00001d034886,1,"304,715,056","304,719,918",0.0339,0.1810,0.1874,arms,Zm00001d012823,5,"925,235","928,518",0.0414,0.1602,0.2585,Sobic.001G007200,arms
4,Zm00001d034885,1,"304,665,327","304,670,409",0.0195,0.1610,0.1211,arms,Zm00001d012827,5,"939,545","943,194",0.0267,0.2088,0.1280,Sobic.001G007600,arms
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4573,Zm00001d014844,5,"65,186,102","65,197,475",0.0231,0.1363,0.1694,peri,Zm00001d036363,6,"85,772,731","85,773,825",0.0308,0.1255,0.2455,Sobic.010G273800,peri
4574,Zm00001d014858,5,"65,680,729","65,683,012",0.0312,0.4094,0.0763,peri,Zm00001d036355,6,"85,387,744","85,388,687",0.0915,0.7903,0.1158,Sobic.010G275200,peri
4575,Zm00001d014867,5,"66,241,452","66,267,159",0.0220,0.1322,0.1664,peri,Zm00001d036351,6,"85,213,378","85,230,505",0.0141,0.1414,0.0994,Sobic.010G275800,peri
4576,Zm00001d014874,5,"66,662,043","66,664,946",0.0370,0.4109,0.0900,peri,Zm00001d036346,6,"85,065,660","85,075,694",0.0248,0.4042,0.0613,Sobic.010G276200,peri


### Evolutionary Metrics Pre-check

In [15]:
import pandas as pd

print("=== Evolutionary Metrics Pre-check ===\n")
print(f"Input dataframe shape: {KaKsO.shape}")
print(f"Columns: {KaKsO.columns.tolist()}\n")

warnings = 0

# Check for duplicates in Maize1 independently
dup_m1 = KaKsO['Maize1'].duplicated()
if dup_m1.any():
    dup_m1_ids = KaKsO.loc[dup_m1, 'Maize1'].unique().tolist()
    raise ValueError(
        f"CRITICAL: {len(dup_m1_ids)} duplicate Maize1 IDs detected in KaKsO file. "
        f"This indicates an upstream processing error that must be corrected before proceeding.\n"
        f"Affected IDs: {dup_m1_ids}"
    )
else:
    print("✓ No duplicate Maize1 IDs detected")

# Check for duplicates in Maize2 independently
dup_m2 = KaKsO['Maize2'].duplicated()
if dup_m2.any():
    dup_m2_ids = KaKsO.loc[dup_m2, 'Maize2'].unique().tolist()
    raise ValueError(
        f"CRITICAL: {len(dup_m2_ids)} duplicate Maize2 IDs detected in KaKsO file. "
        f"This indicates an upstream processing error that must be corrected before proceeding.\n"
        f"Affected IDs: {dup_m2_ids}"
    )
else:
    print("✓ No duplicate Maize2 IDs detected")

# Check for duplicate pairs
dup_pairs = KaKsO.duplicated(subset=['Maize1', 'Maize2'])
if dup_pairs.any():
    dup_pair_rows = KaKsO.loc[dup_pairs, ['Maize1', 'Maize2']]
    raise ValueError(
        f"CRITICAL: {dup_pairs.sum()} duplicate Maize1+Maize2 pair combinations detected. "
        f"This indicates an upstream processing error that must be corrected before proceeding.\n"
        f"Affected pairs:\n{dup_pair_rows}"
    )
else:
    print("✓ No duplicate Maize1+Maize2 pair combinations detected\n")

# Cross-reference pairs against master list
# Check Maize1 coverage
master_m1 = set(maize_df['Maize1'])
file_m1 = set(KaKsO['Maize1'])
absent_m1 = master_m1 - file_m1
if absent_m1:
    print(f"WARNING: {len(absent_m1)} Maize1 genes from master list absent from KaKsO file.")
    print(f"   Affected IDs: {sorted(list(absent_m1))}\n")
    warnings += 1
else:
    print("✓ All Maize1 genes from master list present in KaKsO file")

# Check Maize2 coverage
master_m2 = set(maize_df['Maize2'])
file_m2 = set(KaKsO['Maize2'])
absent_m2 = master_m2 - file_m2
if absent_m2:
    print(f"WARNING: {len(absent_m2)} Maize2 genes from master list absent from KaKsO file.")
    print(f"   Affected IDs: {sorted(list(absent_m2))}\n")
    warnings += 1
else:
    print("✓ All Maize2 genes from master list present in KaKsO file\n")

# Check for NaNs in Ka, Ks, O columns in the input file
kaks_cols = ['Ka_M1', 'Ks_M1', 'O_M1', 'Ka_M2', 'Ks_M2', 'O_M2']
for col in kaks_cols:
    n_nan = KaKsO[col].isna().sum()
    if n_nan > 0:
        affected = KaKsO.loc[KaKsO[col].isna(), ['Maize1', 'Maize2']].values.tolist()
        print(f"WARNING: {n_nan} NaN values in {col} in input file.")
        print(f"   Affected pairs: {affected}\n")
        warnings += 1
    else:
        print(f"✓ No NaN values in {col}")

print(f"\n=== Pre-check complete. Total warnings: {warnings} ===")
print("Review all warnings before proceeding.")

=== Evolutionary Metrics Pre-check ===

Input dataframe shape: (4578, 17)
Columns: ['Maize1', 'Chr._M1', 'Start_M1', 'End_M1', 'Ka_M1', 'Ks_M1', 'O_M1', 'Loc_M1', 'Maize2', 'Chr._M2', 'Start_M2', 'End_M2', 'Ka_M2', 'Ks_M2', 'O_M2', 'Sorghum syntelog', 'Unnamed: 16']

✓ No duplicate Maize1 IDs detected
✓ No duplicate Maize2 IDs detected
✓ No duplicate Maize1+Maize2 pair combinations detected

✓ All Maize1 genes from master list present in KaKsO file
✓ All Maize2 genes from master list present in KaKsO file

✓ No NaN values in Ka_M1
✓ No NaN values in Ks_M1
✓ No NaN values in O_M1
✓ No NaN values in Ka_M2
✓ No NaN values in Ks_M2
✓ No NaN values in O_M2

=== Pre-check complete. Total warnings: 0 ===
Review all warnings before proceeding.


### Evolutionary Metrics Processing and Merge

In [16]:
print("=== Evolutionary Metrics Processing and Merge ===\n")

# Select only the columns needed for the merge
kaks_clean = KaKsO[['Maize1', 'Maize2', 'Ka_M1', 'Ks_M1', 'O_M1', 'Ka_M2', 'Ks_M2', 'O_M2']]

# Single merge on both Maize1 and Maize2 simultaneously
maize_df = pd.merge(maize_df, kaks_clean, on=['Maize1', 'Maize2'], how='left')

# Hard checkpoint
assert len(maize_df) == 4578, f"CRITICAL: Row count is {len(maize_df)}, expected 4578. Silent duplication occurred."
print(f"✓ Row count verified: {len(maize_df)} rows")
print(f"✓ Column count: {len(maize_df.columns)}")

# NaN report
print("\n--- NaN Report ---")
for col in ['Ka_M1', 'Ks_M1', 'O_M1', 'Ka_M2', 'Ks_M2', 'O_M2']:
    n = maize_df[col].isna().sum()
    if n > 0:
        print(f"{col}: {n} NaNs — real missing data, pairs will be dropped at final merge step")
    else:
        print(f"✓  {col}: no NaNs")

=== Evolutionary Metrics Processing and Merge ===

✓ Row count verified: 4578 rows
✓ Column count: 67

--- NaN Report ---
✓  Ka_M1: no NaNs
✓  Ks_M1: no NaNs
✓  O_M1: no NaNs
✓  Ka_M2: no NaNs
✓  Ks_M2: no NaNs
✓  O_M2: no NaNs


In [17]:
display(maize_df)

,Maize1,Location1,Maize2,Location2,Group,H2AZ_down_M1,H3K4me1_down_M1,H3K4me3_down_M1,H3K9ac_down_M1,H3K27ac_down_M1,...,avg_expression_M1,tau_M1,avg_expression_M2,tau_M2,Ka_M1,Ks_M1,O_M1,Ka_M2,Ks_M2,O_M2
0,Zm00001d034914,arms,Zm00001d012815,arms,I,-0.026707,-0.060370,-0.080652,-0.034977,0.055285,...,4.970892,0.238271,3.869726,0.309422,0.0095,0.1597,0.0595,0.0211,0.1668,0.1264
1,Zm00001d034896,arms,Zm00001d012817,arms,I,0.847482,-0.053267,-0.103033,-0.069541,0.054290,...,4.366637,0.231049,4.259741,0.198875,0.1012,0.0651,1.5547,0.0163,0.1557,0.1044
2,Zm00001d034890,arms,Zm00001d012820,arms,I,-0.019175,-0.088925,-0.078468,-0.023988,-0.088444,...,3.351751,0.459761,5.699181,0.238295,0.0397,0.7272,0.0546,0.0300,0.5350,0.0561
3,Zm00001d034886,arms,Zm00001d012823,arms,I,0.060965,0.541438,0.078854,0.019759,0.124945,...,4.239039,0.338206,2.233451,0.468086,0.0339,0.1810,0.1874,0.0414,0.1602,0.2585
4,Zm00001d034885,arms,Zm00001d012827,arms,I,0.145319,0.029047,0.501670,0.089114,0.375345,...,4.151482,0.241467,4.156553,0.339642,0.0195,0.1610,0.1211,0.0267,0.2088,0.1280
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4573,Zm00001d046979,peri,Zm00001d036635,arms,IV,-0.057408,-0.057085,-0.057615,0.035318,-0.057781,...,1.307560,0.441706,0.954048,0.436261,0.0361,0.1167,0.3098,0.0661,0.0628,1.0519
4574,Zm00001d046981,peri,Zm00001d036638,arms,IV,0.000384,-0.023365,0.008214,0.027345,-0.015147,...,0.659109,0.873126,0.242756,0.881164,0.0630,0.1386,0.4543,0.0588,0.1226,0.4800
4575,Zm00001d046986,peri,Zm00001d036623,arms,IV,0.539172,0.068531,0.431359,0.160578,0.490966,...,0.268271,0.917121,0.093041,0.941349,0.0202,0.2303,0.0876,0.0376,0.3480,0.1081
4576,Zm00001d046996,peri,Zm00001d036626,arms,IV,-0.106654,0.559587,0.530210,0.109033,0.229105,...,6.714851,0.151034,6.002171,0.201425,0.0084,0.3106,0.0271,0.0120,0.4487,0.0268


## Recombination Rates

In [18]:
recomb_df = pd.read_csv(DATA_DIR / "Tables20210819.csv", index_col=False)
print(recomb_df.columns)
display(recomb_df)

Index(['gene_id', 'recombination rate'], dtype='object')


,gene_id,recombination rate
0,Zm00001d027230,NaN
1,Zm00001d027231,NaN
2,Zm00001d027232,NaN
3,Zm00001d027233,NaN
4,Zm00001d027234,NaN
...,...,...
39000,Zm00001d048573,NaN
39001,Zm00001d048574,NaN
39002,Zm00001d048575,NaN
39003,Zm00001d048576,NaN


### Recombination Rate Pre-check

In [19]:
import pandas as pd

print("=== Recombination Rate Pre-check ===\n")
print(f"Input dataframe shape: {recomb_df.shape}")
print(f"Columns: {recomb_df.columns.tolist()}\n")

# Hard duplicate check
duplicates = recomb_df['gene_id'].duplicated()
if duplicates.any():
    dup_ids = recomb_df.loc[duplicates, 'gene_id'].unique().tolist()
    raise ValueError(
        f"CRITICAL: {len(dup_ids)} duplicate gene_ids detected in recombination rate file. "
        f"This indicates an upstream processing error that must be corrected before proceeding.\n"
        f"Affected IDs: {dup_ids}"
    )
else:
    print("✓ No duplicate gene_ids detected\n")

# Cross-reference against master list
master_genes = set(maize_df['Maize1']).union(set(maize_df['Maize2']))
file_genes = set(recomb_df['gene_id'])
absent_genes = master_genes - file_genes

if absent_genes:
    print(f"WARNING: {len(absent_genes)} master list genes absent from recombination file.")
    print(f"   These will produce NaN values in the merge and require pair-wise dropout.")
    print(f"   Absent gene IDs: {sorted(list(absent_genes))}\n")
else:
    print("✓ All master list genes present in recombination file\n")

# Check for NaNs in recombination rate column in the input file
nan_recomb = recomb_df['recombination rate'].isna().sum()
if nan_recomb > 0:
    nan_genes = recomb_df.loc[recomb_df['recombination rate'].isna(), 'gene_id'].tolist()
    print(f"WARNING: {nan_recomb} genes have NaN in recombination rate in input file.")
    print(f"   These are true missing values. Pairs will be dropped at final merge step.")
    print()
else:
    print("✓ No NaN values detected in recombination rate column\n")

print("=== Pre-check complete. Review warnings before proceeding. ===")

=== Recombination Rate Pre-check ===

Input dataframe shape: (39005, 2)
Columns: ['gene_id', 'recombination rate']

✓ No duplicate gene_ids detected

✓ All master list genes present in recombination file

   These are true missing values. Pairs will be dropped at final merge step.

=== Pre-check complete. Review warnings before proceeding. ===


### Recombination Rate Processing and Merge

In [20]:
print("=== Recombination Rate Processing and Merge ===\n")

recomb_clean = recomb_df[['gene_id', 'recombination rate']]

# Merge M1
recomb_M1 = recomb_clean.rename(columns={
    'gene_id': 'Maize1',
    'recombination rate': 'recomb_M1'
})
maize_df = pd.merge(maize_df, recomb_M1, on='Maize1', how='left')

# Merge M2
recomb_M2 = recomb_clean.rename(columns={
    'gene_id': 'Maize2',
    'recombination rate': 'recomb_M2'
})
maize_df = pd.merge(maize_df, recomb_M2, on='Maize2', how='left')

# Hard checkpoint
assert len(maize_df) == 4578, f"CRITICAL: Row count is {len(maize_df)}, expected 4578. Silent duplication occurred."
print(f"✓ Row count verified: {len(maize_df)} rows")
print(f"✓ Column count: {len(maize_df.columns)}")

# NaN report
print("\n--- NaN Report ---")
for col in ['recomb_M1', 'recomb_M2']:
    n = maize_df[col].isna().sum()
    if n > 0:
        print(f"{col}: {n} NaNs — true missing data, pairs will be dropped at final merge step")
    else:
        print(f"✓  {col}: no NaNs")

=== Recombination Rate Processing and Merge ===

✓ Row count verified: 4578 rows
✓ Column count: 69

--- NaN Report ---
recomb_M1: 216 NaNs — true missing data, pairs will be dropped at final merge step
recomb_M2: 34 NaNs — true missing data, pairs will be dropped at final merge step


## TE Data

In [21]:
te_distance_df = pd.read_csv(DATA_DIR / "TE_shortest_dis.csv", index_col=False, header=None)
te_density_df = pd.read_csv(DATA_DIR / "TE_density.csv", index_col=False, header=None)

In [22]:
print("te_distance_df")
display(te_distance_df)
print("te_density_df")
display(te_density_df)

te_distance_df


,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,Zm00001d034914,arms,1,305341236,305352529,-,helitron,305353005,305354133,ID=DHH00002B73v401180;Name=DHH00002B73v401180_...,...,arms,5,776419,787466,+,LTR_retrotransposon,754940,773050,ID=RLC00004B73v409865;Name=RLC00004B73v409865_...,3369
1,Zm00001d034896,arms,1,305008203,305016567,+,helitron,305007884,305024246,ID=DHH00008B73v400149;Name=DHH00008B73v400149_...,...,arms,5,834692,841621,-,helitron,838010,857764,ID=DHH00006B73v400523;Name=DHH00006B73v400523_...,0
2,Zm00001d034890,arms,1,304868663,304870630,+,LTR_retrotransposon,304856373,304944337,ID=RLG00085B73v400028;Name=RLG00085B73v400028_...,...,arms,5,848901,851867,-,helitron,838010,857764,ID=DHH00006B73v400523;Name=DHH00006B73v400523_...,0
3,Zm00001d034886,arms,1,304715056,304719918,+,terminal_inverted_repeat_element,304714506,304714750,"ID=DTM_1_304714506_304714750;Name=DTM,TSDlen9,...",...,arms,5,925235,928518,-,LTR_retrotransposon,907751,916800,ID=RLC00002B73v405623;Name=RLC00002B73v405623_...,8435
4,Zm00001d034885,arms,1,304665327,304670409,-,helitron,304667985,304669514,ID=DHH00002B73v406834;Name=DHH00002B73v406834_...,...,arms,5,939545,943194,+,LTR_retrotransposon,907751,916800,ID=RLC00002B73v405623;Name=RLC00002B73v405623_...,22745
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4573,Zm00001d014844,peri,5,65186102,65197475,-,LTR_retrotransposon,65203157,65211919,ID=RLC00002B73v405996;Name=RLC00002B73v405996_...,...,peri,6,85772731,85773825,+,LTR_retrotransposon,85774385,85803187,ID=RLC01680B73v400003;Name=RLC01680B73v400003_...,560
4574,Zm00001d014858,peri,5,65680729,65683012,-,LTR_retrotransposon,65593021,65682839,ID=RLG18788B73v400001;Name=RLG18788B73v400001_...,...,peri,6,85387744,85388687,-,LTR_retrotransposon,85391537,85401002,ID=RLG00001B73v409245;Name=RLG00001B73v409245_...,2850
4575,Zm00001d014867,peri,5,66241452,66267159,-,LTR_retrotransposon,66245132,66250778,ID=RLC00052B73v400069;Name=RLC00052B73v400069_...,...,peri,6,85213378,85230505,+,LTR_retrotransposon,85163388,85254955,ID=RLG16933B73v400001;Name=RLG16933B73v400001_...,0
4576,Zm00001d014874,peri,5,66662043,66664946,-,helitron,66650954,66663218,ID=DHH00044B73v400022;Name=DHH00044B73v400022_...,...,peri,6,85065660,85075694,+,terminal_inverted_repeat_element,85061028,85061135,"ID=DTA_6_85061028_85061135;Name=DTA,TSDlen8,TI...",4525


te_density_df


,0,1,2,3,4,5,6,7,8,9,...,391,392,393,394,395,396,397,398,399,400
0,Zm00001d027230,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0
1,Zm00001d027231,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0
2,Zm00001d027232,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.00,1.00,1.00,1.00,1.0,1.0,1.0,1.0,1.0,1.0
3,Zm00001d027233,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0
4,Zm00001d027234,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39000,Zm00001d048573,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.00,1.00,1.00,1.00,1.0,1.0,1.0,1.0,1.0,1.0
39001,Zm00001d048574,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.00,1.00,1.00,1.00,1.0,1.0,1.0,1.0,1.0,1.0
39002,Zm00001d048575,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.68,0.78,0.88,0.98,1.0,1.0,1.0,1.0,1.0,1.0
39003,Zm00001d048576,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0


### TE Distance Pre-checks

In [23]:
import pandas as pd

# Assign column names
te_distance_df.columns = [
    'Maize1', 'LocationM1', 'ChrM1', 'startM1', 'endM1', 'strandM1',
    'typeM1', 'startTEM1', 'endTEM1', 'idM1', 'distM1', 'Maize2',
    'LocationM2', 'ChrM2', 'startM2', 'endM2', 'strandM2', 'typeM2',
    'startTEM2', 'endTEM2', 'idM2', 'distM2'
]

print("=== TE Distance Pre-check ===\n")
print(f"te_distance_df shape: {te_distance_df.shape}")
warnings = 0

# Duplicate check: Maize1 independently
dup_m1 = te_distance_df['Maize1'].duplicated()
if dup_m1.any():
    dup_m1_ids = te_distance_df.loc[dup_m1, 'Maize1'].unique().tolist()
    raise ValueError(
        f"CRITICAL: {len(dup_m1_ids)} duplicate Maize1 IDs detected in TE distance file. "
        f"Upstream processing error must be corrected before proceeding.\n"
        f"Affected IDs: {dup_m1_ids}"
    )
else:
    print("✓ No duplicate Maize1 IDs detected")

# Duplicate check: Maize2 independently
dup_m2 = te_distance_df['Maize2'].duplicated()
if dup_m2.any():
    dup_m2_ids = te_distance_df.loc[dup_m2, 'Maize2'].unique().tolist()
    raise ValueError(
        f"CRITICAL: {len(dup_m2_ids)} duplicate Maize2 IDs detected in TE distance file. "
        f"Upstream processing error must be corrected before proceeding.\n"
        f"Affected IDs: {dup_m2_ids}"
    )
else:
    print("✓ No duplicate Maize2 IDs detected")

# Duplicate check: pair combination
dup_pairs = te_distance_df.duplicated(subset=['Maize1', 'Maize2'])
if dup_pairs.any():
    dup_pair_rows = te_distance_df.loc[dup_pairs, ['Maize1', 'Maize2']]
    raise ValueError(
        f"CRITICAL: {dup_pairs.sum()} duplicate Maize1+Maize2 pair combinations detected. "
        f"Upstream processing error must be corrected before proceeding.\n"
        f"Affected pairs:\n{dup_pair_rows}"
    )
else:
    print("✓ No duplicate Maize1+Maize2 pair combinations detected\n")

# Cross-reference pairs against master list
master_m1 = set(maize_df['Maize1'])
master_m2 = set(maize_df['Maize2'])
file_m1 = set(te_distance_df['Maize1'])
file_m2 = set(te_distance_df['Maize2'])

absent_m1 = master_m1 - file_m1
if absent_m1:
    print(f"WARNING: {len(absent_m1)} Maize1 genes from master list absent from TE distance file.")
    print(f"   Real missing data — pairs will be dropped at final merge step.")
    print(f"   Absent IDs: {sorted(list(absent_m1))}\n")
    warnings += 1
else:
    print("✓ All Maize1 genes present in TE distance file")

absent_m2 = master_m2 - file_m2
if absent_m2:
    print(f"WARNING: {len(absent_m2)} Maize2 genes from master list absent from TE distance file.")
    print(f"   Real missing data — pairs will be dropped at final merge step.")
    print(f"   Absent IDs: {sorted(list(absent_m2))}\n")
    warnings += 1
else:
    print("✓ All Maize2 genes present in TE distance file\n")

# Check for NaNs in distance and type columns
for col in ['distM1', 'distM2', 'typeM1', 'typeM2']:
    n_nan = te_distance_df[col].isna().sum()
    if n_nan > 0:
        print(f"WARNING: {n_nan} NaN values in {col} in TE distance file.")
        warnings += 1
    else:
        print(f"✓ No NaN values in {col}")

print(f"\n=== TE Distance Pre-check complete. Total warnings: {warnings} ===\n")

=== TE Distance Pre-check ===

te_distance_df shape: (4578, 22)
✓ No duplicate Maize1 IDs detected
✓ No duplicate Maize2 IDs detected
✓ No duplicate Maize1+Maize2 pair combinations detected

✓ All Maize1 genes present in TE distance file
✓ All Maize2 genes present in TE distance file

✓ No NaN values in distM1
✓ No NaN values in distM2
✓ No NaN values in typeM1
✓ No NaN values in typeM2

=== TE Distance Pre-check complete. Total warnings: 0 ===



### TE Density Pre-checks

In [24]:
# Assign column names
te_density_df.columns = ['gene_id'] + [f'col{i}' for i in range(1, len(te_density_df.columns))]

# TE Density Pre-check
print("=== TE Density Pre-check ===\n")
print(f"te_density_df shape: {te_density_df.shape}")
print(f"Expected: 1 gene_id column + 400 window columns\n")

# Validate column count
if len(te_density_df.columns) != 401:
    raise ValueError(
        f"CRITICAL: te_density_df has {len(te_density_df.columns)} columns, expected 401 "
        f"(1 gene_id + 200 upstream + 200 downstream windows)."
    )
else:
    print("✓ Column count verified: 401 columns (1 gene_id + 400 windows)")

# Hard duplicate check
dup_density = te_density_df['gene_id'].duplicated()
if dup_density.any():
    dup_ids = te_density_df.loc[dup_density, 'gene_id'].unique().tolist()
    raise ValueError(
        f"CRITICAL: {len(dup_ids)} duplicate gene_ids detected in TE density file. "
        f"Upstream processing error must be corrected before proceeding.\n"
        f"Affected IDs: {dup_ids}"
    )
else:
    print("✓ No duplicate gene_ids detected\n")

# Cross-reference against master list
master_genes = set(maize_df['Maize1']).union(set(maize_df['Maize2']))
density_genes = set(te_density_df['gene_id'])
absent_density = master_genes - density_genes

if absent_density:
    print(f"WARNING: {len(absent_density)} master list genes absent from TE density file.")
    print(f"   Real missing data — pairs will be dropped at final merge step.")
    print(f"   Absent gene IDs: {sorted(list(absent_density))}\n")
else:
    print("✓ All master list genes present in TE density file\n")

# Validate density values are bounded [0, 1]
density_vals = te_density_df.iloc[:, 1:]
out_of_bounds = ((density_vals < 0) | (density_vals > 1)).any().any()
if out_of_bounds:
    print("WARNING: TE density values outside [0, 1] detected. Cap at 1.0 was not enforced upstream.")
else:
    print("✓ All TE density values are within expected [0, 1] bounds")
    
print(f"\n=== TE Density Pre-check complete. Total warnings: {warnings} ===\n")

=== TE Density Pre-check ===

te_density_df shape: (39005, 401)
Expected: 1 gene_id column + 400 window columns

✓ Column count verified: 401 columns (1 gene_id + 400 windows)
✓ No duplicate gene_ids detected

✓ All master list genes present in TE density file

✓ All TE density values are within expected [0, 1] bounds

=== TE Density Pre-check complete. Total warnings: 0 ===



In [25]:
# Validate TE types against expected categories
expected_types = {'helitron', 'terminal_inverted_repeat_element', 
                  'solo_LTR', 'LTR_retrotransposon', 'SINE_element', 'LINE_element'}
for col in ['typeM1', 'typeM2']:
    observed_types = set(te_distance_df[col].dropna().unique())
    unexpected = observed_types - expected_types
    if unexpected:
        print(f"WARNING: Unexpected TE types in {col}: {unexpected}")
        print(f"   These will not be mapped by the group dictionary and will produce NaN.")
        warnings += 1
    else:
        print(f"✓ All TE types in {col} are within expected categories")

✓ All TE types in typeM1 are within expected categories
✓ All TE types in typeM2 are within expected categories


### TE Processing and Merge

In [26]:
print("=== TE Processing and Merge ===\n")

# --- TE Distance and Categorical Type ---

# Map TE types to group numbers using strict categorical hierarchy
group_dict = {
    'helitron': 1,
    'terminal_inverted_repeat_element': 2,
    'solo_LTR': 3,
    'LTR_retrotransposon': 3,
    'SINE_element': 4,
    'LINE_element': 4
}

te_distance_df['TEGroupM1'] = te_distance_df['typeM1'].map(group_dict)
te_distance_df['TEGroupM2'] = te_distance_df['typeM2'].map(group_dict)

# One-hot encode TE groups using pd.get_dummies()
te_ohe_m1 = pd.get_dummies(te_distance_df['TEGroupM1'], prefix='TEGroupM1').astype(float)
te_ohe_m2 = pd.get_dummies(te_distance_df['TEGroupM2'], prefix='TEGroupM2').astype(float)

# Ensure all four group columns are present even if a category is absent in this dataset
for i in range(1, 5):
    if f'TEGroupM1_{i}' not in te_ohe_m1.columns:
        te_ohe_m1[f'TEGroupM1_{i}'] = 0.0
    if f'TEGroupM2_{i}' not in te_ohe_m2.columns:
        te_ohe_m2[f'TEGroupM2_{i}'] = 0.0

te_ohe_m1 = te_ohe_m1[[f'TEGroupM1_{i}' for i in range(1, 5)]]
te_ohe_m2 = te_ohe_m2[[f'TEGroupM2_{i}' for i in range(1, 5)]]

# Build clean TE distance dataframe for merging
te_merge_df = pd.concat([
    te_distance_df[['Maize1', 'Maize2', 'distM1', 'distM2']],
    te_ohe_m1,
    te_ohe_m2
], axis=1)

print("Clean TE distance dataframe for merging")
display(te_merge_df)

# Single merge on both Maize1 and Maize2 simultaneously
maize_df = pd.merge(maize_df, te_merge_df, on=['Maize1', 'Maize2'], how='left')

print("maize_df after TE distance merge")
display(maize_df)

# Hard checkpoint after TE distance merge
assert len(maize_df) == 4578, f"CRITICAL: Row count is {len(maize_df)}, expected 4578 after TE distance merge."
print(f"✓ Row count verified after TE distance merge: {len(maize_df)} rows")

=== TE Processing and Merge ===

Clean TE distance dataframe for merging


,Maize1,Maize2,distM1,distM2,TEGroupM1_1,TEGroupM1_2,TEGroupM1_3,TEGroupM1_4,TEGroupM2_1,TEGroupM2_2,TEGroupM2_3,TEGroupM2_4
0,Zm00001d034914,Zm00001d012815,476,3369,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,Zm00001d034896,Zm00001d012817,0,0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,Zm00001d034890,Zm00001d012820,0,0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
3,Zm00001d034886,Zm00001d012823,306,8435,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,Zm00001d034885,Zm00001d012827,0,22745,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
4573,Zm00001d014844,Zm00001d036363,5682,560,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4574,Zm00001d014858,Zm00001d036355,0,2850,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4575,Zm00001d014867,Zm00001d036351,0,0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4576,Zm00001d014874,Zm00001d036346,0,4525,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


maize_df after TE distance merge


,Maize1,Location1,Maize2,Location2,Group,H2AZ_down_M1,H3K4me1_down_M1,H3K4me3_down_M1,H3K9ac_down_M1,H3K27ac_down_M1,...,distM1,distM2,TEGroupM1_1,TEGroupM1_2,TEGroupM1_3,TEGroupM1_4,TEGroupM2_1,TEGroupM2_2,TEGroupM2_3,TEGroupM2_4
0,Zm00001d034914,arms,Zm00001d012815,arms,I,-0.026707,-0.060370,-0.080652,-0.034977,0.055285,...,476,3369,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,Zm00001d034896,arms,Zm00001d012817,arms,I,0.847482,-0.053267,-0.103033,-0.069541,0.054290,...,0,0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,Zm00001d034890,arms,Zm00001d012820,arms,I,-0.019175,-0.088925,-0.078468,-0.023988,-0.088444,...,0,0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
3,Zm00001d034886,arms,Zm00001d012823,arms,I,0.060965,0.541438,0.078854,0.019759,0.124945,...,306,8435,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,Zm00001d034885,arms,Zm00001d012827,arms,I,0.145319,0.029047,0.501670,0.089114,0.375345,...,0,22745,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4573,Zm00001d046979,peri,Zm00001d036635,arms,IV,-0.057408,-0.057085,-0.057615,0.035318,-0.057781,...,857,246,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
4574,Zm00001d046981,peri,Zm00001d036638,arms,IV,0.000384,-0.023365,0.008214,0.027345,-0.015147,...,3075,0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4575,Zm00001d046986,peri,Zm00001d036623,arms,IV,0.539172,0.068531,0.431359,0.160578,0.490966,...,2995,659,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4576,Zm00001d046996,peri,Zm00001d036626,arms,IV,-0.106654,0.559587,0.530210,0.109033,0.229105,...,5244,1452,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


✓ Row count verified after TE distance merge: 4578 rows


In [27]:
# --- TE Density ---

# Upstream: col1 through col200
# Downstream: col201 through col400
upstream_cols = [f'col{i}' for i in range(1, 201)]
downstream_cols = [f'col{i}' for i in range(201, 401)]

te_density_df['TEdenseAvgUp'] = te_density_df[upstream_cols].mean(axis=1)
te_density_df['TEdenseMaxUp'] = te_density_df[upstream_cols].max(axis=1)
te_density_df['TEdenseAvgDown'] = te_density_df[downstream_cols].mean(axis=1)
te_density_df['TEdenseMaxDown'] = te_density_df[downstream_cols].max(axis=1)

density_clean = te_density_df[['gene_id', 'TEdenseAvgUp', 'TEdenseMaxUp', 
                                 'TEdenseAvgDown', 'TEdenseMaxDown']]

# Merge M1
density_M1 = density_clean.rename(columns={
    'gene_id': 'Maize1',
    'TEdenseAvgUp': 'TEdenseAvgUp_M1',
    'TEdenseMaxUp': 'TEdenseMaxUp_M1',
    'TEdenseAvgDown': 'TEdenseAvgDown_M1',
    'TEdenseMaxDown': 'TEdenseMaxDown_M1'
})
maize_df = pd.merge(maize_df, density_M1, on='Maize1', how='left')

# Merge M2
density_M2 = density_clean.rename(columns={
    'gene_id': 'Maize2',
    'TEdenseAvgUp': 'TEdenseAvgUp_M2',
    'TEdenseMaxUp': 'TEdenseMaxUp_M2',
    'TEdenseAvgDown': 'TEdenseAvgDown_M2',
    'TEdenseMaxDown': 'TEdenseMaxDown_M2'
})
maize_df = pd.merge(maize_df, density_M2, on='Maize2', how='left')
display(maize_df)

,Maize1,Location1,Maize2,Location2,Group,H2AZ_down_M1,H3K4me1_down_M1,H3K4me3_down_M1,H3K9ac_down_M1,H3K27ac_down_M1,...,TEGroupM2_3,TEGroupM2_4,TEdenseAvgUp_M1,TEdenseMaxUp_M1,TEdenseAvgDown_M1,TEdenseMaxDown_M1,TEdenseAvgUp_M2,TEdenseMaxUp_M2,TEdenseAvgDown_M2,TEdenseMaxDown_M2
0,Zm00001d034914,arms,Zm00001d012815,arms,I,-0.026707,-0.060370,-0.080652,-0.034977,0.055285,...,1.0,0.0,0.77935,1.00,0.6885,1.00,0.5396,1.00,0.2840,1.0
1,Zm00001d034896,arms,Zm00001d012817,arms,I,0.847482,-0.053267,-0.103033,-0.069541,0.054290,...,0.0,0.0,0.00000,0.00,0.0000,0.00,0.1189,1.00,0.6081,1.0
2,Zm00001d034890,arms,Zm00001d012820,arms,I,-0.019175,-0.088925,-0.078468,-0.023988,-0.088444,...,0.0,0.0,0.00000,0.00,0.0000,0.00,0.5610,1.00,0.5720,1.0
3,Zm00001d034886,arms,Zm00001d012823,arms,I,0.060965,0.541438,0.078854,0.019759,0.124945,...,1.0,0.0,0.19170,1.00,0.0000,0.00,0.0695,1.00,0.0860,1.0
4,Zm00001d034885,arms,Zm00001d012827,arms,I,0.145319,0.029047,0.501670,0.089114,0.375345,...,1.0,0.0,0.60105,1.00,0.0000,0.00,0.0765,1.00,0.0000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4573,Zm00001d046979,peri,Zm00001d036635,arms,IV,-0.057408,-0.057085,-0.057615,0.035318,-0.057781,...,0.0,0.0,0.00000,0.00,0.0345,0.69,0.1730,1.00,0.0000,0.0
4574,Zm00001d046981,peri,Zm00001d036638,arms,IV,0.000384,-0.023365,0.008214,0.027345,-0.015147,...,1.0,0.0,0.03550,0.71,0.6215,1.00,0.0164,0.76,0.9705,1.0
4575,Zm00001d046986,peri,Zm00001d036623,arms,IV,0.539172,0.068531,0.431359,0.160578,0.490966,...,1.0,0.0,0.34640,1.00,0.0000,0.00,0.3400,1.00,0.6480,1.0
4576,Zm00001d046996,peri,Zm00001d036626,arms,IV,-0.106654,0.559587,0.530210,0.109033,0.229105,...,1.0,0.0,0.49805,1.00,0.1610,1.00,0.5565,1.00,0.0815,1.0


In [28]:
# Hard checkpoint after TE density merge
assert len(maize_df) == 4578, f"CRITICAL: Row count is {len(maize_df)}, expected 4578 after TE density merge."
print(f"✓ Row count verified after TE density merge: {len(maize_df)} rows")
print(f"✓ Column count: {len(maize_df.columns)}")

# NaN report
print("\n--- NaN Report ---")
te_cols = ['distM1', 'distM2',
           'TEGroupM1_1', 'TEGroupM1_2', 'TEGroupM1_3', 'TEGroupM1_4',
           'TEGroupM2_1', 'TEGroupM2_2', 'TEGroupM2_3', 'TEGroupM2_4',
           'TEdenseAvgUp_M1', 'TEdenseMaxUp_M1', 'TEdenseAvgDown_M1', 'TEdenseMaxDown_M1',
           'TEdenseAvgUp_M2', 'TEdenseMaxUp_M2', 'TEdenseAvgDown_M2', 'TEdenseMaxDown_M2']

for col in te_cols:
    n = maize_df[col].isna().sum()
    if n > 0:
        print(f"{col}: {n} NaNs — real missing data, pairs will be dropped at final merge step")
    else:
        print(f"✓  {col}: no NaNs")

✓ Row count verified after TE density merge: 4578 rows
✓ Column count: 87

--- NaN Report ---
✓  distM1: no NaNs
✓  distM2: no NaNs
✓  TEGroupM1_1: no NaNs
✓  TEGroupM1_2: no NaNs
✓  TEGroupM1_3: no NaNs
✓  TEGroupM1_4: no NaNs
✓  TEGroupM2_1: no NaNs
✓  TEGroupM2_2: no NaNs
✓  TEGroupM2_3: no NaNs
✓  TEGroupM2_4: no NaNs
✓  TEdenseAvgUp_M1: no NaNs
✓  TEdenseMaxUp_M1: no NaNs
✓  TEdenseAvgDown_M1: no NaNs
✓  TEdenseMaxDown_M1: no NaNs
✓  TEdenseAvgUp_M2: no NaNs
✓  TEdenseMaxUp_M2: no NaNs
✓  TEdenseAvgDown_M2: no NaNs
✓  TEdenseMaxDown_M2: no NaNs


## ACR Data

In [29]:
acr_data = pd.read_csv('/blue/meixiazhao/laylaaschuster/maizeGDM/final_maizeMLpreprocess/input_ACR/Maize_ACR_features_updated.tsv',
                      sep='\t', index_col=False)
print(acr_data.columns)
display(acr_data)

Index(['gene_id', 'summit_fold_enrichment', 'upstream_distance',
       'downstream_distance'],
      dtype='object')


,gene_id,summit_fold_enrichment,upstream_distance,downstream_distance
0,Zm00001d027230,21.57540,18817.0,5850.0
1,Zm00001d027231,21.57540,29.0,25405.0
2,Zm00001d027232,21.57540,114574.0,36611.0
3,Zm00001d027233,21.57540,91396.0,55967.0
4,Zm00001d027234,21.57540,89969.0,62995.0
...,...,...,...,...
39000,Zm00001d048573,6.23758,272010.0,275897.0
39001,Zm00001d048574,19.68450,211572.0,315486.0
39002,Zm00001d048575,19.68450,45364.0,470277.0
39003,Zm00001d048576,19.68450,25008.0,520316.0


### ACR Pre-checks 

In [30]:
import pandas as pd

print("=== ACR Data Pre-check ===\n")
print(f"Input dataframe shape: {acr_data.shape}")
print(f"Columns: {acr_data.columns.tolist()}\n")

warnings = 0

# Hard duplicate check
duplicates = acr_data['gene_id'].duplicated()
if duplicates.any():
    dup_ids = acr_data.loc[duplicates, 'gene_id'].unique().tolist()
    raise ValueError(
        f"CRITICAL: {len(dup_ids)} duplicate gene_ids detected in ACR file. "
        f"Upstream processing error must be corrected before proceeding.\n"
        f"Affected IDs: {dup_ids}"
    )
else:
    print("✓ No duplicate gene_ids detected\n")

# Cross-reference against master list
master_genes = set(maize_df['Maize1']).union(set(maize_df['Maize2']))
file_genes = set(acr_data['gene_id'])
absent_genes = master_genes - file_genes

if absent_genes:
    print(f"WARNING: {len(absent_genes)} master list genes absent from ACR file.")
    print(f"   These genes and their homoeologs will require pair-wise dropout.")
    print(f"   Absent gene IDs: {sorted(list(absent_genes))}\n")
    warnings += 1
else:
    print("✓ All master list genes present in ACR file\n")

# Check summit_fold_enrichment for NaNs — none expected
nan_summit = acr_data['summit_fold_enrichment'].isna().sum()
if nan_summit > 0:
    nan_summit_genes = acr_data.loc[acr_data['summit_fold_enrichment'].isna(), 'gene_id'].tolist()
    print(f"WARNING: {nan_summit} NaN values in summit_fold_enrichment.")
    print(f"   Affected gene IDs: {nan_summit_genes}\n")
    warnings += 1
else:
    print("✓ No NaN values in summit_fold_enrichment\n")

# Check distance columns for NaNs — chromosome edge NaNs are acceptable
for col in ['upstream_distance', 'downstream_distance']:
    n_nan = acr_data[col].isna().sum()
    if n_nan > 0:
        nan_genes = acr_data.loc[acr_data[col].isna(), 'gene_id'].tolist()
        print(f"INFO: {n_nan} NaN values in {col}.")
        print(f"   These are expected chromosome-edge cases and will be RETAINED.")
        print(f"   Affected gene IDs: {nan_genes}\n")
    else:
        print(f"✓ No NaN values in {col}\n")

print(f"=== Pre-check complete. Total warnings: {warnings} ===")

=== ACR Data Pre-check ===

Input dataframe shape: (39005, 4)
Columns: ['gene_id', 'summit_fold_enrichment', 'upstream_distance', 'downstream_distance']

✓ No duplicate gene_ids detected

✓ All master list genes present in ACR file

✓ No NaN values in summit_fold_enrichment

INFO: 13 NaN values in upstream_distance.
   These are expected chromosome-edge cases and will be RETAINED.
   Affected gene IDs: ['Zm00001d007980', 'Zm00001d039233', 'Zm00001d044716', 'Zm00001d039226', 'Zm00001d039228', 'Zm00001d018593', 'Zm00001d018594', 'Zm00001d018595', 'Zm00001d022647', 'Zm00001d012791', 'Zm00001d012792', 'Zm00001d044719', 'Zm00001d044723']

INFO: 16 NaN values in downstream_distance.
   These are expected chromosome-edge cases and will be RETAINED.
   Affected gene IDs: ['Zm00001d001763', 'Zm00001d001765', 'Zm00001d039234', 'Zm00001d044714', 'Zm00001d044715', 'Zm00001d048578', 'Zm00001d012794', 'Zm00001d039229', 'Zm00001d039230', 'Zm00001d039231', 'Zm00001d018591', 'Zm00001d022648', 'Zm00001d

### ACR Processing and Merge

In [31]:
print("=== ACR Data Processing and Merge ===\n")

acr_clean = acr_data[['gene_id', 'summit_fold_enrichment', 
                        'upstream_distance', 'downstream_distance']]

# Merge M1
acr_M1 = acr_clean.rename(columns={
    'gene_id': 'Maize1',
    'summit_fold_enrichment': 'acr_S_M1',
    'upstream_distance': 'acr_Lup_M1',
    'downstream_distance': 'acr_Ldown_M1'
})
maize_df = pd.merge(maize_df, acr_M1, on='Maize1', how='left')

# Merge M2
acr_M2 = acr_clean.rename(columns={
    'gene_id': 'Maize2',
    'summit_fold_enrichment': 'acr_S_M2',
    'upstream_distance': 'acr_Lup_M2',
    'downstream_distance': 'acr_Ldown_M2'
})
maize_df = pd.merge(maize_df, acr_M2, on='Maize2', how='left')

# Hard checkpoint
assert len(maize_df) == 4578, f"CRITICAL: Row count is {len(maize_df)}, expected 4578. Silent duplication occurred."
print(f"✓ Row count verified: {len(maize_df)} rows")
print(f"✓ Column count: {len(maize_df.columns)}")

# NaN report
print("\n--- NaN Report ---")
for col in ['acr_S_M1', 'acr_S_M2']:
    n = maize_df[col].isna().sum()
    if n > 0:
        print(f"{col}: {n} NaNs — unexpected, requires investigation")
    else:
        print(f"✓  {col}: no NaNs")

for col in ['acr_Lup_M1', 'acr_Ldown_M1', 'acr_Lup_M2', 'acr_Ldown_M2']:
    n = maize_df[col].isna().sum()
    if n > 0:
        print(f"{col}: {n} NaNs — chromosome-edge cases, biologically meaningful, RETAINED")
    else:
        print(f"✓  {col}: no NaNs")

=== ACR Data Processing and Merge ===

✓ Row count verified: 4578 rows
✓ Column count: 93

--- NaN Report ---
✓  acr_S_M1: no NaNs
✓  acr_S_M2: no NaNs
acr_Lup_M1: 1 NaNs — chromosome-edge cases, biologically meaningful, RETAINED
acr_Ldown_M1: 2 NaNs — chromosome-edge cases, biologically meaningful, RETAINED
acr_Lup_M2: 1 NaNs — chromosome-edge cases, biologically meaningful, RETAINED
✓  acr_Ldown_M2: no NaNs


#### Note: NaNs Retained
NaNs retained for upstream and downstream ACR distances: where a gene sits at a chromosome edge with no upstream or downstream flanking sequence. This is a real genomic constraint.

## Methylation data

### Methylation Pre-checks

In [33]:
avg_files = [
    '/blue/meixiazhao/laylaaschuster/maizeGDM/final_maizeMLpreprocess/input_DNAmethyl/methylation_rep12.csv',
    '/blue/meixiazhao/laylaaschuster/maizeGDM/final_maizeMLpreprocess/input_DNAmethyl/methylation_rep24.csv'
]

max_files = [
    '/blue/meixiazhao/laylaaschuster/maizeGDM/final_maizeMLpreprocess/input_DNAmethyl/methylation_max_rep12.csv',
    '/blue/meixiazhao/laylaaschuster/maizeGDM/final_maizeMLpreprocess/input_DNAmethyl/methylation_max_rep24.csv'
]

In [34]:
import pandas as pd

def get_avg_gene_ids(file_path):
    gene_ids = []
    with open(file_path, 'r', encoding='utf-8-sig') as f:
        for line in f:
            parts = line.strip().split(',')
            if parts[0]:
                gene_ids.append(parts[0])
    return set(gene_ids)

def get_max_gene_ids(file_path):
    gene_ids = []
    with open(file_path, 'r', encoding='utf-8-sig') as f:
        for line in f:
            parts = line.strip().split(',')
            if parts[0]:
                gene_ids.append(parts[0].split(':')[1])
    return set(gene_ids)

print("=== Methylation Data Pre-check ===\n")
master_genes = set(maize_df['Maize1']).union(set(maize_df['Maize2']))
warnings = 0

# Check average files
print("--- Average Methylation Files ---")
for file_path in avg_files:
    file_name = file_path.split('/')[-1]
    gene_ids = get_avg_gene_ids(file_path)
    
    # Duplicate check
    all_ids = []
    with open(file_path, 'r', encoding='utf-8-sig') as f:
        for line in f:
            parts = line.strip().split(',')
            if parts[0]:
                all_ids.append(parts[0])
    if len(all_ids) != len(set(all_ids)):
        raise ValueError(
            f"CRITICAL: Duplicate gene IDs detected in {file_name}. "
            f"Upstream processing error must be corrected before proceeding."
        )
    else:
        print(f"✓ {file_name}: no duplicate gene IDs")
    
    # Coverage check
    absent = master_genes - gene_ids
    if absent:
        print(f"{file_name}: {len(absent)} master list genes absent")
        print(f"   Absent IDs: {sorted(list(absent))}")
        warnings += 1
    else:
        print(f"✓ {file_name}: all master list genes present")

# Check max files
print("\n--- Maximum Methylation Files ---")
for file_path in max_files:
    file_name = file_path.split('/')[-1]
    gene_ids = get_max_gene_ids(file_path)
    
    absent = master_genes - gene_ids
    if absent:
        print(f"{file_name}: {len(absent)} master list genes absent")
        print(f"   Absent IDs: {sorted(list(absent))}")
        warnings += 1
    else:
        print(f"✓ {file_name}: all master list genes present")

print(f"\n=== Pre-check complete. Total warnings: {warnings} ===")
print("NaNs from absent or incomplete genes are real missing data requiring pair-wise dropout.")

=== Methylation Data Pre-check ===

--- Average Methylation Files ---
✓ methylation_rep12.csv: no duplicate gene IDs
methylation_rep12.csv: 2 master list genes absent
   Absent IDs: ['Zm00001d008951', 'Zm00001d020647']
✓ methylation_rep24.csv: no duplicate gene IDs
methylation_rep24.csv: 2 master list genes absent
   Absent IDs: ['Zm00001d008951', 'Zm00001d020647']

--- Maximum Methylation Files ---
methylation_max_rep12.csv: 2 master list genes absent
   Absent IDs: ['Zm00001d008951', 'Zm00001d020647']
methylation_max_rep24.csv: 2 master list genes absent
   Absent IDs: ['Zm00001d008951', 'Zm00001d020647']

=== Pre-check complete. Total warnings: 4 ===
NaNs from absent or incomplete genes are real missing data requiring pair-wise dropout.


### Methylation Processing and Merge

In [35]:
print("=== Methylation Processing and Merge ===\n")

def parse_methyl_avg_file(file_path):
    """Parse average methylation file into wide dataframe indexed by gene_id."""
    records = {}
    with open(file_path, 'r', encoding='utf-8-sig') as f:
        for line in f:
            parts = line.strip().split(',')
            gene_id = parts[0]
            record = {}
            for part in parts[1:]:
                if '_' in part:
                    key, value = part.rsplit('_', 1)
                    record[key] = float(value)
            records[gene_id] = record
    return pd.DataFrame.from_dict(records, orient='index')

def parse_methyl_max_file(file_path):
    """Parse max methylation file into wide dataframe indexed by gene_id.
    Body rows are skipped entirely."""
    records = {}
    with open(file_path, 'r', encoding='utf-8-sig') as f:
        for line in f:
            parts = line.strip().split(',')
            gene_id = parts[0].split(':')[1]
            region = parts[2]
            
            # Skip body rows — max body methylation excluded per README
            if region == 'body':
                continue
            
            meth_type = parts[1]
            value = float(parts[-2])
            key = f"{meth_type}_{region}_max"
            
            if gene_id not in records:
                records[gene_id] = {}
            records[gene_id][key] = value
    return pd.DataFrame.from_dict(records, orient='index')

# --- Average methylation ---
# Parse both replicates and average using groupby mean
# Genes present in only one replicate use that single value
# Features missing from both replicates for a gene remain NaN naturally
avg_rep12 = parse_methyl_avg_file(avg_files[0])
avg_rep24 = parse_methyl_avg_file(avg_files[1])
methyl_avg = pd.concat([avg_rep12, avg_rep24]).groupby(level=0).mean()

# Verify expected columns
expected_avg_cols = {'CHH_up', 'CHH_down', 'CHH_body',
                     'CHG_up', 'CHG_down', 'CHG_body',
                     'CG_up', 'CG_down', 'CG_body'}
missing_avg_cols = expected_avg_cols - set(methyl_avg.columns)
if missing_avg_cols:
    raise ValueError(f"CRITICAL: Missing expected average methylation columns: {missing_avg_cols}")
else:
    print(f"✓ Average methylation columns verified: {sorted(methyl_avg.columns.tolist())}")

# --- Maximum methylation (flanking only) ---
max_rep12 = parse_methyl_max_file(max_files[0])
max_rep24 = parse_methyl_max_file(max_files[1])
methyl_max = pd.concat([max_rep12, max_rep24]).groupby(level=0).mean()

# Verify expected columns — body max columns must NOT be present
expected_max_cols = {'CHH_up_max', 'CHH_down_max',
                     'CHG_up_max', 'CHG_down_max',
                     'CG_up_max', 'CG_down_max'}
missing_max_cols = expected_max_cols - set(methyl_max.columns)
body_max_cols = [c for c in methyl_max.columns if 'body' in c]
if missing_max_cols:
    raise ValueError(f"CRITICAL: Missing expected max methylation columns: {missing_max_cols}")
if body_max_cols:
    raise ValueError(f"CRITICAL: Body max methylation columns present but must be excluded: {body_max_cols}")
else:
    print(f"✓ Maximum methylation columns verified: {sorted(methyl_max.columns.tolist())}")
    print(f"✓ No body max methylation columns present\n")

# Combine avg and max into single methylation dataframe
methyl_combined = methyl_avg.join(methyl_max, how='outer')
methyl_combined = methyl_combined.reset_index().rename(columns={'index': 'gene_id'})
print(f"Combined methylation dataframe shape: {methyl_combined.shape}")
print(f"Expected: 15 feature columns + 1 gene_id column\n")

# Merge M1
methyl_M1 = methyl_combined.rename(columns={
    'gene_id': 'Maize1',
    'CHH_up': 'M1_CHH_up', 'CHH_down': 'M1_CHH_down', 'CHH_body': 'M1_CHH_body',
    'CHG_up': 'M1_CHG_up', 'CHG_down': 'M1_CHG_down', 'CHG_body': 'M1_CHG_body',
    'CG_up': 'M1_CG_up', 'CG_down': 'M1_CG_down', 'CG_body': 'M1_CG_body',
    'CHH_up_max': 'M1_CHH_up_max', 'CHH_down_max': 'M1_CHH_down_max',
    'CHG_up_max': 'M1_CHG_up_max', 'CHG_down_max': 'M1_CHG_down_max',
    'CG_up_max': 'M1_CG_up_max', 'CG_down_max': 'M1_CG_down_max'
})
maize_df = pd.merge(maize_df, methyl_M1, on='Maize1', how='left')

# Merge M2
methyl_M2 = methyl_combined.rename(columns={
    'gene_id': 'Maize2',
    'CHH_up': 'M2_CHH_up', 'CHH_down': 'M2_CHH_down', 'CHH_body': 'M2_CHH_body',
    'CHG_up': 'M2_CHG_up', 'CHG_down': 'M2_CHG_down', 'CHG_body': 'M2_CHG_body',
    'CG_up': 'M2_CG_up', 'CG_down': 'M2_CG_down', 'CG_body': 'M2_CG_body',
    'CHH_up_max': 'M2_CHH_up_max', 'CHH_down_max': 'M2_CHH_down_max',
    'CHG_up_max': 'M2_CHG_up_max', 'CHG_down_max': 'M2_CHG_down_max',
    'CG_up_max': 'M2_CG_up_max', 'CG_down_max': 'M2_CG_down_max'
})
maize_df = pd.merge(maize_df, methyl_M2, on='Maize2', how='left')

# Hard checkpoint
assert len(maize_df) == 4578, f"CRITICAL: Row count is {len(maize_df)}, expected 4578."
print(f"✓ Row count verified: {len(maize_df)} rows")
print(f"✓ Column count: {len(maize_df.columns)}")

# NaN report
print("\n--- NaN Report ---")
methyl_cols = [c for c in maize_df.columns if c.startswith('M1_') or c.startswith('M2_')]
nan_report = maize_df[methyl_cols].isna().sum()
nan_report = nan_report[nan_report > 0]
if len(nan_report) > 0:
    print(f"NaNs detected in methylation columns — real missing data, pairs will be dropped at final merge step:")
    print(nan_report)
else:
    print("✓ No NaNs in methylation columns")

=== Methylation Processing and Merge ===

✓ Average methylation columns verified: ['CG_body', 'CG_down', 'CG_up', 'CHG_body', 'CHG_down', 'CHG_up', 'CHH_body', 'CHH_down', 'CHH_up']
✓ Maximum methylation columns verified: ['CG_down_max', 'CG_up_max', 'CHG_down_max', 'CHG_up_max', 'CHH_down_max', 'CHH_up_max']
✓ No body max methylation columns present

Combined methylation dataframe shape: (24608, 16)
Expected: 15 feature columns + 1 gene_id column

✓ Row count verified: 4578 rows
✓ Column count: 123

--- NaN Report ---
NaNs detected in methylation columns — real missing data, pairs will be dropped at final merge step:
M1_CHH_up           11
M1_CHH_down          7
M1_CHH_body        543
M1_CHG_up           11
M1_CHG_down          7
M1_CHG_body        578
M1_CG_up            11
M1_CG_down           7
M1_CG_body         577
M1_CHG_up_max       11
M1_CHH_up_max       11
M1_CG_up_max        11
M1_CHH_down_max      7
M1_CG_down_max       7
M1_CHG_down_max      7
M2_CHH_up            8
M2_CHH

In [36]:
display(maize_df)

,Maize1,Location1,Maize2,Location2,Group,H2AZ_down_M1,H3K4me1_down_M1,H3K4me3_down_M1,H3K9ac_down_M1,H3K27ac_down_M1,...,M2_CHG_body,M2_CG_up,M2_CG_down,M2_CG_body,M2_CHG_up_max,M2_CHH_up_max,M2_CG_up_max,M2_CHH_down_max,M2_CG_down_max,M2_CHG_down_max
0,Zm00001d034914,arms,Zm00001d012815,arms,I,-0.026707,-0.060370,-0.080652,-0.034977,0.055285,...,0.010,0.880,0.685,0.755,1.00000,0.25960,1.0,0.72220,1.0,1.00000
1,Zm00001d034896,arms,Zm00001d012817,arms,I,0.847482,-0.053267,-0.103033,-0.069541,0.054290,...,0.000,0.895,0.755,0.505,1.00000,0.24205,1.0,0.29165,1.0,1.00000
2,Zm00001d034890,arms,Zm00001d012820,arms,I,-0.019175,-0.088925,-0.078468,-0.023988,-0.088444,...,0.150,0.780,0.550,0.480,1.00000,0.24995,1.0,0.60230,1.0,0.87500
3,Zm00001d034886,arms,Zm00001d012823,arms,I,0.060965,0.541438,0.078854,0.019759,0.124945,...,0.010,0.470,0.345,0.380,0.89305,0.40000,1.0,0.02330,1.0,0.08870
4,Zm00001d034885,arms,Zm00001d012827,arms,I,0.145319,0.029047,0.501670,0.089114,0.375345,...,0.005,0.720,0.835,0.790,1.00000,0.35420,1.0,0.07605,1.0,1.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4573,Zm00001d046979,peri,Zm00001d036635,arms,IV,-0.057408,-0.057085,-0.057615,0.035318,-0.057781,...,0.250,0.910,0.110,0.680,1.00000,0.16665,1.0,0.35155,1.0,0.60715
4574,Zm00001d046981,peri,Zm00001d036638,arms,IV,0.000384,-0.023365,0.008214,0.027345,-0.015147,...,0.520,0.275,0.910,0.705,0.97035,0.60715,1.0,0.45040,1.0,1.00000
4575,Zm00001d046986,peri,Zm00001d036623,arms,IV,0.539172,0.068531,0.431359,0.160578,0.490966,...,NaN,0.655,0.830,NaN,0.97380,0.83330,1.0,0.73570,1.0,1.00000
4576,Zm00001d046996,peri,Zm00001d036626,arms,IV,-0.106654,0.559587,0.530210,0.109033,0.229105,...,0.000,0.890,0.655,0.000,1.00000,0.79285,1.0,0.58330,1.0,0.92500


### Checkpoint Feature Dataframe

In [37]:
maize_df.to_csv('/blue/meixiazhao/laylaaschuster/maizeGDM/final_maizeMLpreprocess/checkpoint_maize_df_prior_dropna.csv', 
                index=False)

## Combine M1 and M2 
Combine column-wise and drop NA

In [38]:
print("=== Pair-wise Dropout ===\n")
print(f"Pairs before dropout: {len(maize_df)}")

# Define columns where NaN is explicitly permitted
nan_exempt_cols = {
    'tau_M1', 'tau_M2',
    'acr_Lup_M1', 'acr_Ldown_M1',
    'acr_Lup_M2', 'acr_Ldown_M2'
}

# All feature columns excluding identifiers and exempt columns
dropout_cols = [
    col for col in maize_df.columns
    if col not in {'Maize1', 'Maize2', 'Location1', 'Location2', 'Group'}
    and col not in nan_exempt_cols
]

# Identify pairs where either member has a NaN in any required column
pairs_with_missing = maize_df[dropout_cols].isna().any(axis=1)

# Report which columns are driving dropout before removing rows
if pairs_with_missing.any():
    missing_cols = maize_df.loc[pairs_with_missing, dropout_cols].isna().sum()
    missing_cols = missing_cols[missing_cols > 0].sort_values(ascending=False)
    print(f"{pairs_with_missing.sum()} pairs flagged for dropout due to missing data.")
    print(f"\nColumns contributing to dropout:")
    for col, count in missing_cols.items():
        print(f"  {col}: {count} affected pairs")
else:
    print("✓ No pairs flagged for dropout")

# Drop affected pairs
maize_df = maize_df[~pairs_with_missing].reset_index(drop=True)

print(f"\n✓ Pairs after dropout: {len(maize_df)}")
print(f"✓ Pairs removed: {pairs_with_missing.sum()}")
assert len(maize_df) > 0, "CRITICAL: All pairs were dropped. Review dropout column list."

=== Pair-wise Dropout ===

Pairs before dropout: 4578
1042 pairs flagged for dropout due to missing data.

Columns contributing to dropout:
  M2_CHG_body: 588 affected pairs
  M2_CG_body: 586 affected pairs
  M1_CHG_body: 578 affected pairs
  M1_CG_body: 577 affected pairs
  M2_CHH_body: 557 affected pairs
  M1_CHH_body: 543 affected pairs
  recomb_M1: 216 affected pairs
  recomb_M2: 34 affected pairs
  M1_CG_up_max: 11 affected pairs
  M1_CHH_up_max: 11 affected pairs
  M1_CHG_up_max: 11 affected pairs
  M1_CG_up: 11 affected pairs
  M1_CHG_up: 11 affected pairs
  M1_CHH_up: 11 affected pairs
  M2_CHH_up_max: 8 affected pairs
  M2_CG_up_max: 8 affected pairs
  M2_CHG_up_max: 8 affected pairs
  M2_CHH_up: 8 affected pairs
  M2_CHG_up: 8 affected pairs
  M2_CG_up: 8 affected pairs
  M2_CG_down_max: 7 affected pairs
  M2_CG_down: 7 affected pairs
  M1_CHG_down_max: 7 affected pairs
  M1_CG_down_max: 7 affected pairs
  M1_CHH_down_max: 7 affected pairs
  M1_CG_down: 7 affected pairs
  M1_

### Concatenate: Wide to Long Format

In [39]:
# Concatenate M1 and M2 vertically

# Metadata
concat_ids = pd.concat([maize_df['Maize1'], maize_df['Maize2']], ignore_index=True)
concat_loc = pd.concat([maize_df['Location1'], maize_df['Location2']], ignore_index=True)

# GC Content
concat_genicGC = pd.concat([maize_df['GC_genic_M1'], maize_df['GC_genic_M2']], ignore_index=True)
concat_promGC = pd.concat([maize_df['GC_prom_M1'], maize_df['GC_prom_M2']], ignore_index=True)

# Expression
concat_exp = pd.concat([maize_df['avg_expression_M1'], maize_df['avg_expression_M2']], ignore_index=True)
concat_tau = pd.concat([maize_df['tau_M1'], maize_df['tau_M2']], ignore_index=True)

# Evolutionary Metrics
concat_ka = pd.concat([maize_df['Ka_M1'], maize_df['Ka_M2']], ignore_index=True)
concat_ks = pd.concat([maize_df['Ks_M1'], maize_df['Ks_M2']], ignore_index=True)
concat_omgea = pd.concat([maize_df['O_M1'], maize_df['O_M2']], ignore_index=True)

# Recombination
concat_recomb = pd.concat([maize_df['recomb_M1'], maize_df['recomb_M2']], ignore_index=True)

# ACRs
concat_acr_Lup = pd.concat([maize_df['acr_Lup_M1'], maize_df['acr_Lup_M2']], ignore_index=True)
concat_acr_Ldown = pd.concat([maize_df['acr_Ldown_M1'], maize_df['acr_Ldown_M2']], ignore_index=True)
concat_acr_S = pd.concat([maize_df['acr_S_M1'], maize_df['acr_S_M2']], ignore_index=True)

# Methylation avg
concat_up_CHH = pd.concat([maize_df['M1_CHH_up'], maize_df['M2_CHH_up']], ignore_index=True)
concat_down_CHH = pd.concat([maize_df['M1_CHH_down'], maize_df['M2_CHH_down']], ignore_index=True)
concat_genic_CHH = pd.concat([maize_df['M1_CHH_body'], maize_df['M2_CHH_body']], ignore_index=True)

concat_up_CHG = pd.concat([maize_df['M1_CHG_up'], maize_df['M2_CHG_up']], ignore_index=True)
concat_down_CHG = pd.concat([maize_df['M1_CHG_down'], maize_df['M2_CHG_down']], ignore_index=True)
concat_genic_CHG = pd.concat([maize_df['M1_CHG_body'], maize_df['M2_CHG_body']], ignore_index=True)

concat_up_CG = pd.concat([maize_df['M1_CG_up'], maize_df['M2_CG_up']], ignore_index=True)
concat_down_CG = pd.concat([maize_df['M1_CG_down'], maize_df['M2_CG_down']], ignore_index=True)
concat_genic_CG = pd.concat([maize_df['M1_CG_body'], maize_df['M2_CG_body']], ignore_index=True)

# Methylation max
concat_up_CHH_max = pd.concat([maize_df['M1_CHH_up_max'], maize_df['M2_CHH_up_max']], ignore_index=True)
concat_down_CHH_max = pd.concat([maize_df['M1_CHH_down_max'], maize_df['M2_CHH_down_max']], ignore_index=True)

concat_up_CHG_max = pd.concat([maize_df['M1_CHG_up_max'], maize_df['M2_CHG_up_max']], ignore_index=True)
concat_down_CHG_max = pd.concat([maize_df['M1_CHG_down_max'], maize_df['M2_CHG_down_max']], ignore_index=True)

concat_up_CG_max = pd.concat([maize_df['M1_CG_up_max'], maize_df['M2_CG_up_max']], ignore_index=True)
concat_down_CG_max = pd.concat([maize_df['M1_CG_down_max'], maize_df['M2_CG_down_max']], ignore_index=True)

# TE Distance and Density (abundance)
concat_TEdist = pd.concat([maize_df['distM1'], maize_df['distM2']], ignore_index=True)
concat_TEdenseAvgUp = pd.concat([maize_df['TEdenseAvgUp_M1'], maize_df['TEdenseAvgUp_M2']], ignore_index=True)
concat_TEdenseAvgDown = pd.concat([maize_df['TEdenseAvgDown_M1'], maize_df['TEdenseAvgDown_M2']], ignore_index=True)
concat_TEdenseMaxUp = pd.concat([maize_df['TEdenseMaxUp_M1'], maize_df['TEdenseMaxUp_M2']], ignore_index=True)
concat_TEdenseMaxDown = pd.concat([maize_df['TEdenseMaxDown_M1'], maize_df['TEdenseMaxDown_M2']], ignore_index=True)
# TE Groups
concat_TE1 = pd.concat([maize_df['TEGroupM1_1'], maize_df['TEGroupM2_1']], ignore_index=True)
concat_TE2 = pd.concat([maize_df['TEGroupM1_2'], maize_df['TEGroupM2_2']], ignore_index=True)
concat_TE3 = pd.concat([maize_df['TEGroupM1_3'], maize_df['TEGroupM2_3']], ignore_index=True)
concat_TE4 = pd.concat([maize_df['TEGroupM1_4'], maize_df['TEGroupM2_4']], ignore_index=True)

# Histones
concat_H2AZ_down = pd.concat([maize_df['H2AZ_down_M1'], maize_df['H2AZ_down_M2']], ignore_index=True)
concat_H3K4me1_down = pd.concat([maize_df['H3K4me1_down_M1'], maize_df['H3K4me1_down_M2']], ignore_index=True)
concat_H3K4me3_down = pd.concat([maize_df['H3K4me3_down_M1'], maize_df['H3K4me3_down_M2']], ignore_index=True)
concat_H3K9ac_down = pd.concat([maize_df['H3K9ac_down_M1'], maize_df['H3K9ac_down_M2']], ignore_index=True)
concat_H3K27ac_down = pd.concat([maize_df['H3K27ac_down_M1'], maize_df['H3K27ac_down_M2']], ignore_index=True)
concat_H3K27me3_down = pd.concat([maize_df['H3K27me3_down_M1'], maize_df['H3K27me3_down_M2']], ignore_index=True)
concat_H3K36me3_down = pd.concat([maize_df['H3K36me3_down_M1'], maize_df['H3K36me3_down_M2']], ignore_index=True)
concat_H3K56ac_down = pd.concat([maize_df['H3K56ac_down_M1'], maize_df['H3K56ac_down_M2']], ignore_index=True)

concat_H2AZ_genebody = pd.concat([maize_df['H2AZ_genebody_M1'], maize_df['H2AZ_genebody_M2']], ignore_index=True)
concat_H3K4me1_genebody = pd.concat([maize_df['H3K4me1_genebody_M1'], maize_df['H3K4me1_genebody_M2']], ignore_index=True)
concat_H3K4me3_genebody = pd.concat([maize_df['H3K4me3_genebody_M1'], maize_df['H3K4me3_genebody_M2']], ignore_index=True)
concat_H3K9ac_genebody = pd.concat([maize_df['H3K9ac_genebody_M1'], maize_df['H3K9ac_genebody_M2']], ignore_index=True)
concat_H3K27ac_genebody = pd.concat([maize_df['H3K27ac_genebody_M1'], maize_df['H3K27ac_genebody_M2']], ignore_index=True)
concat_H3K27me3_genebody = pd.concat([maize_df['H3K27me3_genebody_M1'], maize_df['H3K27me3_genebody_M2']], ignore_index=True)
concat_H3K36me3_genebody = pd.concat([maize_df['H3K36me3_genebody_M1'], maize_df['H3K36me3_genebody_M2']], ignore_index=True)
concat_H3K56ac_genebody = pd.concat([maize_df['H3K56ac_genebody_M1'], maize_df['H3K56ac_genebody_M2']], ignore_index=True)

concat_H2AZ_up = pd.concat([maize_df['H2AZ_up_M1'], maize_df['H2AZ_up_M2']], ignore_index=True)
concat_H3K4me1_up = pd.concat([maize_df['H3K4me1_up_M1'], maize_df['H3K4me1_up_M2']], ignore_index=True)
concat_H3K4me3_up = pd.concat([maize_df['H3K4me3_up_M1'], maize_df['H3K4me3_up_M2']], ignore_index=True)
concat_H3K9ac_up = pd.concat([maize_df['H3K9ac_up_M1'], maize_df['H3K9ac_up_M2']], ignore_index=True)
concat_H3K27ac_up = pd.concat([maize_df['H3K27ac_up_M1'], maize_df['H3K27ac_up_M2']], ignore_index=True)
concat_H3K27me3_up = pd.concat([maize_df['H3K27me3_up_M1'], maize_df['H3K27me3_up_M2']], ignore_index=True)
concat_H3K36me3_up = pd.concat([maize_df['H3K36me3_up_M1'], maize_df['H3K36me3_up_M2']], ignore_index=True)
concat_H3K56ac_up = pd.concat([maize_df['H3K56ac_up_M1'], maize_df['H3K56ac_up_M2']], ignore_index=True)

group_stacked = pd.concat([maize_df['Group']] * 2, ignore_index=True)

In [40]:
df_concat = pd.DataFrame({
    'Maize': concat_ids,
    'location': concat_loc,
    'group': group_stacked,

    'GC_genic': concat_genicGC,
    'GC_prom': concat_promGC,
    
    'avg_expression': concat_exp,
    'tau': concat_tau,
    
    'Ka': concat_ka,
    'Ks': concat_ks,
    'O': concat_omgea,

    'acr_Lup': concat_acr_Lup,
    'acr_Ldown': concat_acr_Ldown,
    'acr_S': concat_acr_S,
    
    'recomb': concat_recomb,
    
    'CHH_up_avg': concat_up_CHH,
    'CHH_down_avg': concat_down_CHH,
    'CHH_body_avg': concat_genic_CHH,
    'CHG_up_avg': concat_up_CHG,
    'CHG_down_avg': concat_down_CHG,
    'CHG_body_avg': concat_genic_CHG,
    'CG_up_avg': concat_up_CG,
    'CG_down_avg': concat_down_CG,
    'CG_body_avg': concat_genic_CG,
    
    'CHH_up_max': concat_up_CHH_max,
    'CHH_down_max': concat_down_CHH_max,
    'CHG_up_max': concat_up_CHG_max,
    'CHG_down_max': concat_down_CHG_max,
    'CG_up_max': concat_up_CG_max,
    'CG_down_max': concat_down_CG_max,
    
    'TEdist': concat_TEdist,
    'TEdenseAvgUp': concat_TEdenseAvgUp,
    'TEdenseAvgDown': concat_TEdenseAvgDown,
    'TEdenseMaxUp': concat_TEdenseMaxUp,
    'TEdenseMaxDown': concat_TEdenseMaxDown,

    'TE1': concat_TE1,
    'TE2': concat_TE2,
    'TE3': concat_TE3,
    'TE4': concat_TE4,
    
    'H2AZ_down': concat_H2AZ_down,
    'H3K4me1_down': concat_H3K4me1_down,
    'H3K4me3_down': concat_H3K4me3_down,
    'H3K9ac_down': concat_H3K9ac_down,
    'H3K27ac_down': concat_H3K27ac_down,
    'H3K27me3_down': concat_H3K27me3_down,
    'H3K36me3_down': concat_H3K36me3_down,
    'H3K56ac_down': concat_H3K56ac_down,
    
    'H2AZ_genebody': concat_H2AZ_genebody,
    'H3K4me1_genebody': concat_H3K4me1_genebody,
    'H3K4me3_genebody': concat_H3K4me3_genebody,
    'H3K9ac_genebody': concat_H3K9ac_genebody,
    'H3K27ac_genebody': concat_H3K27ac_genebody,
    'H3K27me3_genebody': concat_H3K27me3_genebody,
    'H3K36me3_genebody': concat_H3K36me3_genebody,
    'H3K56ac_genebody': concat_H3K56ac_genebody,
    
    'H2AZ_up': concat_H2AZ_up,
    'H3K4me1_up': concat_H3K4me1_up,
    'H3K4me3_up': concat_H3K4me3_up,
    'H3K9ac_up': concat_H3K9ac_up,
    'H3K27ac_up': concat_H3K27ac_up,
    'H3K27me3_up': concat_H3K27me3_up,
    'H3K36me3_up': concat_H3K36me3_up,
    'H3K56ac_up': concat_H3K56ac_up,
})

df_concat['location'] = df_concat['location'].replace({'arms': 1.0, 'peri': 0.0})

display(df_concat)

,Maize,location,group,GC_genic,GC_prom,avg_expression,tau,Ka,Ks,O,...,H3K36me3_genebody,H3K56ac_genebody,H2AZ_up,H3K4me1_up,H3K4me3_up,H3K9ac_up,H3K27ac_up,H3K27me3_up,H3K36me3_up,H3K56ac_up
0,Zm00001d034876,1.0,I,0.502865,0.57,3.235371,0.327983,0.0147,0.1950,0.0754,...,1.039540,0.259413,0.021806,-0.118156,-0.077072,-0.039742,-0.076416,0.022874,-0.117635,-0.099811
1,Zm00001d034871,1.0,I,0.494667,0.63,3.881011,0.160152,0.0066,0.1733,0.0382,...,1.521993,0.198370,0.059901,-0.055869,-0.011277,0.005502,0.126622,0.014895,-0.039157,0.052127
2,Zm00001d034869,1.0,I,0.600000,0.65,0.228264,0.939040,0.0695,0.2654,0.2618,...,-0.066927,0.148688,0.241938,-0.062509,-0.073585,-0.071346,-0.013039,0.247950,-0.131793,-0.082181
3,Zm00001d034867,1.0,I,0.582043,0.51,4.818114,0.120378,0.0484,0.2641,0.1832,...,1.190428,0.543737,-0.010671,-0.025882,0.001996,0.034852,0.064355,-0.003417,-0.014366,0.011795
4,Zm00001d034862,1.0,I,0.536571,0.29,1.926035,0.363008,0.0387,0.1448,0.2669,...,1.199487,0.247423,-0.036643,-0.029219,0.031193,-0.004799,0.018924,-0.005501,-0.029320,0.064210
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7067,Zm00001d036631,1.0,IV,0.421525,0.46,3.863004,0.198116,0.0160,0.1358,0.1179,...,1.450231,0.202770,-0.066254,-0.092212,-0.052951,-0.004850,-0.011838,-0.048444,-0.103021,-0.103666
7068,Zm00001d036632,1.0,IV,0.510185,0.60,3.184460,0.253859,0.0585,0.1329,0.4401,...,1.388968,0.124384,0.017986,-0.048203,-0.006427,0.054340,0.115300,0.041164,-0.051024,0.020336
7069,Zm00001d036635,1.0,IV,0.428288,0.55,0.954048,0.436261,0.0661,0.0628,1.0519,...,1.833603,0.474707,0.011068,-0.005385,-0.019787,0.046278,0.009533,0.015416,-0.014929,0.003539
7070,Zm00001d036638,1.0,IV,0.579276,0.44,0.242756,0.881164,0.0588,0.1226,0.4800,...,-0.097733,0.029449,0.666150,-0.045887,0.008584,0.086127,0.189805,1.090263,-0.107230,0.087639


In [41]:
df_concat.to_csv('/blue/meixiazhao/laylaaschuster/maizeGDM/final_maizeMLpreprocess/checkpoint2_maize_df_prior_dropna.csv', 
                index=False)

In [42]:
# Verify no gene ID appears in both Maize1 and Maize2 columns
m1_genes = set(maize_df['Maize1'])
m2_genes = set(maize_df['Maize2'])
overlap = m1_genes.intersection(m2_genes)

if overlap:
    raise ValueError(
        f"CRITICAL: {len(overlap)} gene IDs appear in both Maize1 and Maize2 columns. "
        f"WGD label assignment will be incorrect.\n"
        f"Affected IDs: {sorted(list(overlap))}"
    )
else:
    print(f"✓ No gene ID overlap between Maize1 and Maize2 columns. WGD label assignment is safe to proceed.")

✓ No gene ID overlap between Maize1 and Maize2 columns. WGD label assignment is safe to proceed.


In [43]:
# Add WGD label column

# Create a combined dictionary mapping 'Maize1' values to 0 and 'Maize2' values to 1
dict_combined = {key: 0 for key in maize_df['Maize1']} | {key: 1 for key in maize_df['Maize2']}

# Create a new 'WGD' column by mapping the keys from the combined dictionary
df_concat['WGD'] = df_concat['Maize'].map(dict_combined).values

print(df_concat.shape)

(7072, 63)


In [44]:
print(df_concat.shape)
print(df_concat.columns)
df_concat.to_csv('/blue/meixiazhao/laylaaschuster/maizeGDM/final_maizeMLpreprocess/final_featuresets/processedMaize_all_columns_final.csv', index=False)

df_featureset = df_concat.drop(['Maize', 'group'], axis=1)
display(df_featureset)

df_featureset.to_csv('/blue/meixiazhao/laylaaschuster/maizeGDM/final_maizeMLpreprocess/final_featuresets/processedMaize_final.csv', index=False)

(7072, 63)
Index(['Maize', 'location', 'group', 'GC_genic', 'GC_prom', 'avg_expression',
       'tau', 'Ka', 'Ks', 'O', 'acr_Lup', 'acr_Ldown', 'acr_S', 'recomb',
       'CHH_up_avg', 'CHH_down_avg', 'CHH_body_avg', 'CHG_up_avg',
       'CHG_down_avg', 'CHG_body_avg', 'CG_up_avg', 'CG_down_avg',
       'CG_body_avg', 'CHH_up_max', 'CHH_down_max', 'CHG_up_max',
       'CHG_down_max', 'CG_up_max', 'CG_down_max', 'TEdist', 'TEdenseAvgUp',
       'TEdenseAvgDown', 'TEdenseMaxUp', 'TEdenseMaxDown', 'TE1', 'TE2', 'TE3',
       'TE4', 'H2AZ_down', 'H3K4me1_down', 'H3K4me3_down', 'H3K9ac_down',
       'H3K27ac_down', 'H3K27me3_down', 'H3K36me3_down', 'H3K56ac_down',
       'H2AZ_genebody', 'H3K4me1_genebody', 'H3K4me3_genebody',
       'H3K9ac_genebody', 'H3K27ac_genebody', 'H3K27me3_genebody',
       'H3K36me3_genebody', 'H3K56ac_genebody', 'H2AZ_up', 'H3K4me1_up',
       'H3K4me3_up', 'H3K9ac_up', 'H3K27ac_up', 'H3K27me3_up', 'H3K36me3_up',
       'H3K56ac_up', 'WGD'],
      dtype='object')


,location,GC_genic,GC_prom,avg_expression,tau,Ka,Ks,O,acr_Lup,acr_Ldown,...,H3K56ac_genebody,H2AZ_up,H3K4me1_up,H3K4me3_up,H3K9ac_up,H3K27ac_up,H3K27me3_up,H3K36me3_up,H3K56ac_up,WGD
0,1.0,0.502865,0.57,3.235371,0.327983,0.0147,0.1950,0.0754,4170.0,2927.0,...,0.259413,0.021806,-0.118156,-0.077072,-0.039742,-0.076416,0.022874,-0.117635,-0.099811,0
1,1.0,0.494667,0.63,3.881011,0.160152,0.0066,0.1733,0.0382,1.0,59957.0,...,0.198370,0.059901,-0.055869,-0.011277,0.005502,0.126622,0.014895,-0.039157,0.052127,0
2,1.0,0.600000,0.65,0.228264,0.939040,0.0695,0.2654,0.2618,65793.0,213.0,...,0.148688,0.241938,-0.062509,-0.073585,-0.071346,-0.013039,0.247950,-0.131793,-0.082181,0
3,1.0,0.582043,0.51,4.818114,0.120378,0.0484,0.2641,0.1832,63.0,171.0,...,0.543737,-0.010671,-0.025882,0.001996,0.034852,0.064355,-0.003417,-0.014366,0.011795,0
4,1.0,0.536571,0.29,1.926035,0.363008,0.0387,0.1448,0.2669,8693.0,874.0,...,0.247423,-0.036643,-0.029219,0.031193,-0.004799,0.018924,-0.005501,-0.029320,0.064210,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7067,1.0,0.421525,0.46,3.863004,0.198116,0.0160,0.1358,0.1179,102466.0,930.0,...,0.202770,-0.066254,-0.092212,-0.052951,-0.004850,-0.011838,-0.048444,-0.103021,-0.103666,1
7068,1.0,0.510185,0.60,3.184460,0.253859,0.0585,0.1329,0.4401,36.0,103605.0,...,0.124384,0.017986,-0.048203,-0.006427,0.054340,0.115300,0.041164,-0.051024,0.020336,1
7069,1.0,0.428288,0.55,0.954048,0.436261,0.0661,0.0628,1.0519,85844.0,1573.0,...,0.474707,0.011068,-0.005385,-0.019787,0.046278,0.009533,0.015416,-0.014929,0.003539,1
7070,1.0,0.579276,0.44,0.242756,0.881164,0.0588,0.1226,0.4800,15912.0,93129.0,...,0.029449,0.666150,-0.045887,0.008584,0.086127,0.189805,1.090263,-0.107230,0.087639,1


## Subset Featureset into Groups

In [45]:
# Subset Groups

# Verify all group labels are expected values
expected_groups = {'I', 'II', 'III', 'IV'}
observed_groups = set(df_concat['group'].unique())
unexpected_groups = observed_groups - expected_groups
if unexpected_groups:
    raise ValueError(
        f"CRITICAL: Unexpected group labels detected: {unexpected_groups}. "
        f"Review group assignment before subsetting."
    )
else:
    print(f"✓ Group labels verified: {sorted(observed_groups)}\n")

# Verify all genes are assigned to a group
missing_group = df_concat['group'].isna().sum()
if missing_group > 0:
    raise ValueError(
        f"CRITICAL: {missing_group} genes have no group assignment. "
        f"Review WGD label and group assignment steps."
    )

# Subset and save each group
group_sizes = df_concat['group'].value_counts().sort_index()
print("Gene counts per group:")
for group, count in group_sizes.items():
    print(f"  Group {group}: {count} genes ({count // 2} pairs)")

# Verify group counts sum to total
assert group_sizes.sum() == len(df_concat), \
    f"CRITICAL: Group sizes sum to {group_sizes.sum()}, expected {len(df_concat)}."
print(f"\n✓ Group sizes sum verified: {group_sizes.sum()} total genes\n")

# Feature columns only — drop Maize ID and group label
id_cols = ['Maize', 'group']

for group_label in ['I', 'II', 'III', 'IV']:
    df_group = df_concat[df_concat['group'] == group_label].copy()
    df_group_features = df_group.drop(columns=id_cols)
    
    # Verify WGD balance within group — each group should have equal M1 and M2
    wgd_counts = df_group['WGD'].value_counts()
    if wgd_counts.get(0) != wgd_counts.get(1):
        print(f"WARNING: Group {group_label} has unequal WGD counts: "
              f"WGD=0: {wgd_counts.get(0)}, WGD=1: {wgd_counts.get(1)}")
    else:
        print(f"✓ Group {group_label}: {len(df_group)} genes, "
              f"WGD balanced ({wgd_counts.get(0)} per subgenome)")
    
    # Save full version with IDs for traceability
    df_group.to_csv(
        f'/blue/meixiazhao/laylaaschuster/maizeGDM/final_maizeMLpreprocess/final_featuresets/processedMaize_group{group_label}_all_columns_final.csv',
        index=False
    )
    
    # Save feature-only version for model input
    df_group_features.to_csv(
        f'/blue/meixiazhao/laylaaschuster/maizeGDM/final_maizeMLpreprocess/final_featuresets/processedMaize_group{group_label}_final.csv',
        index=False
    )
    
print("\n✓ All group files saved successfully.")

✓ Group labels verified: ['I', 'II', 'III', 'IV']

Gene counts per group:
  Group I: 4292 genes (2146 pairs)
  Group II: 724 genes (362 pairs)
  Group III: 1156 genes (578 pairs)
  Group IV: 900 genes (450 pairs)

✓ Group sizes sum verified: 7072 total genes

✓ Group I: 4292 genes, WGD balanced (2146 per subgenome)
✓ Group II: 724 genes, WGD balanced (362 per subgenome)
✓ Group III: 1156 genes, WGD balanced (578 per subgenome)
✓ Group IV: 900 genes, WGD balanced (450 per subgenome)

✓ All group files saved successfully.
